# Final Project B - The Most Dangerous Streets of Chicago

A data story on where, when, and why traffic crashes happen in Chicago, moving from citywide patterns to street-level risk.

For a guide to run this notebook, please see the Appendix at the end. 

In [1]:
### Loading data & Imports
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


df = pd.read_csv("final.csv", index_col=0)
df_1 = df.copy()

/var/folders/fm/5lbdpfj928d6sqhy_19cp6ww0000gn/T/ipykernel_61357/1297571850.py:7: DtypeWarning: Columns (0: SEX, 1: CELL_PHONE_USE, 2: CRASH_DATE_EST_I, 3: ALIGNMENT, 4: WORK_ZONE_I, 5: NUM_UNITS, 6: MANEUVER, 7: VEHICLE_DEFECT) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("final.csv", index_col=0)


In [2]:
### Styling functions
def rgb_to_rgba(rgb_str, opacity):
    """Convert 'rgb(r, g, b)' with 0–1 floats to 'rgba(R, G, B, alpha)'."""
    vals = rgb_str.replace("rgb(", "").replace(")", "").split(",")
    r, g, b = [round(float(v.strip()) * 255) for v in vals]
    return f"rgba({r}, {g}, {b}, {opacity})"

colors = {
    "BLUE":   "rgb(0.125, 0.314, 0.733)",
    "ORANGE": "rgb(0.988, 0.510, 0.341)",
    "TEAL":   "rgb(0.216, 0.494, 0.498)",
    "PURPLE": "rgb(0.463, 0.082, 0.733)",
    "BROWN":  "rgb(0.651, 0.314, 0.118)",
    "YELLOW": "rgb(0.969,0.812,0.239)",
    "GREEN":  "rgb(0.357, 0.729, 0.608)",
    "GRAY":   "rgb(0.439, 0.463, 0.521)",
}

special_opacity = {
    colors["ORANGE"]: 0.8,
    colors["GREEN"]: 0.8,
}

colors_rgba = {
    name: rgb_to_rgba(rgb, special_opacity.get(rgb, 0.6))
    for name, rgb in colors.items()
}

BLUE   = colors_rgba["BLUE"]
ORANGE = colors_rgba["ORANGE"]
TEAL   = colors_rgba["TEAL"]
PURPLE = colors_rgba["PURPLE"]
BROWN  = colors_rgba["BROWN"]
YELLOW = colors_rgba["YELLOW"]
GREEN  = colors_rgba["GREEN"]
GRAY   = colors_rgba["GRAY"]

GLOBAL_OPACITY = 0.8

LABEL_COLOR = "#262626"
FONT = "Tiempos Text Font"

def _simple_layout(fig, title, height=620, barmode=None):
    fig.update_layout(
        template="none",
        title=dict(text=title, x=0.01, xanchor="left"),
        font=dict(family=FONT, size=18, color=LABEL_COLOR),
        margin=dict(l=60, r=30, t=80, b=90),
        height=height,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    if barmode is not None:
        fig.update_layout(barmode=barmode)
    
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", linecolor="#D4D4D4", tickcolor="#D4D4D4", linewidth=2, title_font=dict(color=LABEL_COLOR))
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", linecolor="#D4D4D4", tickcolor="#D4D4D4", linewidth=2, title_font=dict(color=LABEL_COLOR))

# 1. Motivation

Traffic crashes are a routine part of city life, but their consequences are not distributed evenly. Some corridors see mostly minor collisions, while others produce a disproportionate share of severe injuries and fatalities. This project focuses on severity to reveal where the true human cost concentrates — not just where collisions are common.

---

## What is the dataset?

The analysis uses four City of Chicago open-data sources, all available through the [City of Chicago Data Portal](https://data.cityofchicago.org/) and covering records from **2015 to the present**:

| Dataset | Unit of observation | Key fields |
|---|---|---|
| **Traffic Crashes — Crashes** | One row per crash event | Location, date/time, weather, lighting, control device, contributing cause |
| **Traffic Crashes — People** | One row per person involved | Person type, injury severity, age, action taken |
| **Traffic Crashes — Vehicles** | One row per vehicle involved | Vehicle type, maneuver, defect |
| **Street Center Lines** | One row per street segment | Street name, type, functional class |

After joining on `CRASH_RECORD_ID`, the combined file contains several hundred thousand rows representing all recorded crash events across the city. The crash-level table (deduplicated to one row per crash) and the person-level table are used separately depending on the analysis.

---

## Why these datasets?

- **Completeness and coverage.** The Chicago open-data crash records are citywide, consistently formatted since 2015, and updated regularly. They are among the most detailed municipal crash datasets publicly available in the United States.
- **Multi-table depth.** The separate Crashes, People, and Vehicles tables allow analysis at different levels of granularity: crash-level for street comparisons, person-level for user-group breakdowns.
- **Street-level joining.** The Street Center Lines layer allows crash counts to be attributed to named corridors, which is the core spatial unit of this story.
- **No viable alternative.** There is no traffic-volume dataset available at the same street-level granularity for Chicago. This is a known limitation (see Discussion), but the crash data alone is sufficient to study *relative* severity across corridors.

---

## End-user experience goal

The experience is designed as a **zoom-in narrative** — what Segel and Heer (2010) call a *Martini Glass* structure:

1. **Opening funnel** — citywide patterns establish baseline risk and give readers the broad context.
2. **Narrowing focus** — the analysis moves to the top 15 most dangerous corridors.
3. **Street-level detail** — timing, environmental conditions, and behavioral factors explain *why* specific streets are dangerous.

The goal is to leave readers with a clear, actionable sense that dangerous streets are dangerous **for different reasons** — which points toward targeted, street-specific interventions rather than blanket policies.

# 2. Basic stats and Preprocessing

The snapshot above summarizes the dataset at the crash level. We use a crash-level table (one row per crash) for most street-level summaries and a person-level view when we need exposure by road-user group.

### Cleaning and preprocessing choices

Working with four joined tables introduced substantial complexity. Key decisions are documented here because they affect every downstream figure.

**Deduplication.** The combined file has one row per person per crash. For crash-level analysis we deduplicate on `CRASH_RECORD_ID`, keeping one row per crash event. This is essential before computing any per-street aggregation.

**Missing values.** The data contains significant missingness in several fields:
- `STREET_NAME` is missing for a small fraction of records; those rows are excluded from street-level summaries.
- `INJURIES_FATAL` and `INJURIES_INCAPACITATING` are treated as zero when missing, on the assumption that absence of injury coding implies no severe injury rather than unknown status. This is a simplifying assumption and may slightly undercount severity in older records.
- Categorical fields like `LIGHTING_CONDITION`, `WEATHER_CONDITION`, and `PRIM_CONTRIBUTORY_CAUSE` have substantial "UNKNOWN" categories. We retain these as an explicit category rather than imputing, because the proportion of unknowns is itself informative.

**Type coercion.** `CRASH_YEAR` and `CRASH_HOUR` are coerced to numeric. Rows with unparseable year values are dropped from time-series plots (a very small fraction of the total). `CRASH_HOUR` is clipped to the 0–23 range.

**Severity definition.** A crash is classified as *severe* if it has at least one fatal or incapacitating injury (`INJURIES_FATAL > 0` or `INJURIES_INCAPACITATING > 0`). This binary flag is the core outcome variable throughout the analysis.

**Primary road-user labelling.** Because many crashes involve multiple people, we assign a single priority label per crash for composition charts: Pedestrian > Cyclist > Passenger > Driver. This reflects vulnerability ordering. When full exposure distributions are needed, the person-level table is used directly.

### How we assign a primary road user per crash

A single crash can involve multiple people (drivers, passengers, pedestrians, cyclists). The People table is person-level, but many figures in this notebook are crash-level, so we need one consistent label per crash.

We assign a primary affected group using a priority rule that reflects vulnerability:

* If any pedestrian is involved, the crash is tagged as Pedestrian.
* Else if any cyclist is involved, it is tagged as Cyclist.
* Else if any passenger is involved, it is tagged as Passenger.
* Otherwise it is tagged as Driver.

This keeps comparisons consistent across time and across streets. When we need full exposure counts or distributions, we use the person-level table directly rather than the primary label.

In [3]:
# Basic dataset snapshot (crash-level)
crash_base_basic = df.drop_duplicates(subset=["CRASH_RECORD_ID"]).copy()
crash_base_basic["CRASH_YEAR"] = pd.to_numeric(crash_base_basic["CRASH_YEAR"], errors="coerce")
crash_base_basic["is_severe_crash"] = (
    (crash_base_basic["INJURIES_FATAL"].fillna(0) > 0)
    | (crash_base_basic["INJURIES_INCAPACITATING"].fillna(0) > 0)
)

basic_stats = pd.DataFrame({
    "metric": [
        "Rows in combined file",
        "Unique crashes",
        "Unique people",
        "Year range (min)",
        "Year range (max)",
    ],
    "value": [
        len(df),
        crash_base_basic["CRASH_RECORD_ID"].nunique(),
        df["PERSON_ID"].nunique() if "PERSON_ID" in df.columns else "n/a",
        int(crash_base_basic["CRASH_YEAR"].min()) if crash_base_basic["CRASH_YEAR"].notna().any() else "n/a",
        int(crash_base_basic["CRASH_YEAR"].max()) if crash_base_basic["CRASH_YEAR"].notna().any() else "n/a",
    ],
})

basic_stats

,metric,value
0,Rows in combined file,2280182
1,Unique crashes,1034515
2,Unique people,2275705
3,Year range (min),2013
4,Year range (max),2026


# 3. Data Analysis

### Exploratory Analysis — What the Initial Plots Show

The four plots below establish the baseline context for the rest of the analysis.

In [4]:
### Yearly totals vs severe crashes (citywide)
yearly_basic = (
    crash_base_basic.dropna(subset=["CRASH_YEAR"]) 
    .groupby("CRASH_YEAR", as_index=False)
    .agg(
        total_crashes=("CRASH_RECORD_ID", "nunique"),
        severe_crashes=("is_severe_crash", "sum"),
    )
    .sort_values("CRASH_YEAR")
)
yearly_basic["severe_rate_pct"] = (
    yearly_basic["severe_crashes"] / yearly_basic["total_crashes"] * 100
).round(2)

yearly_basic.tail(10)

fig_basic = make_subplots(specs=[[{"secondary_y": True}]])
fig_basic.add_trace(
    go.Bar(
        x=yearly_basic["CRASH_YEAR"],
        y=yearly_basic["severe_crashes"],
        text=yearly_basic["severe_crashes"],
        texttemplate="%{text:.3s}",
        textposition="outside",
        cliponaxis=False,
        name="Severe crashes",
        marker_color=ORANGE,
        opacity=0.8,
    ),
    secondary_y=False,
)
fig_basic.add_trace(
    go.Scatter(
        x=yearly_basic["CRASH_YEAR"],
        y=yearly_basic["total_crashes"],
        name="Total crashes",
        mode="lines+markers",
        line=dict(color=PURPLE, width=2, dash="dashdot"),
    ),
    secondary_y=True,
)

_simple_layout(fig_basic, "", height=500)
fig_basic.update_layout(
    title=dict(text="<b>Citywide Crash Volume vs. Severe Crashes (Yearly)</b>", x=0.5, xanchor="center"),
    legend=dict(orientation="h", yanchor="top", y=-0.25, xanchor="center", x=0.5),
    margin=dict(l=80, r=40, t=70, b=80),
)
fig_basic.update_yaxes(title_text="Severe crashes", secondary_y=False)
fig_basic.update_yaxes(title_text="Total crashes", secondary_y=True, zeroline=False)
fig_basic.show()

**Citywide crash volume vs. severe crashes (yearly bar + line chart)**  

Total crashes fluctuate between roughly 25 k and 35 k per year, with a visible dip around 2020 that aligns with COVID-19 mobility restrictions. Severe crashes (bars, left axis) track the same general shape but at a much lower scale. The severe fraction sits consistently in the 3–5 % range, confirming that most crashes are minor — but the minority that are severe constitute the real human cost. The relative stability of the severe *rate* across years motivates the street-level focus: if citywide rates are flat, the meaningful variation must be hiding at the corridor level.

In [5]:
### Crash Distribution by Hour of Day
hour_dist = crash_base_basic["CRASH_HOUR"].value_counts().sort_index()
fig_hour = go.Figure(
    data=[
        go.Bar(
            x=hour_dist.index,
            y=hour_dist.values,
            text=hour_dist.values,
            texttemplate="%{text:.3s}",
            textposition="outside",
            cliponaxis=False,
            marker_color=ORANGE,
            opacity=GLOBAL_OPACITY,
        )
    ]
)
_simple_layout(fig_hour, "Crash Distribution by Hour of Day", height=500)
fig_hour.update_layout(
    title=dict(x=0.5, xanchor="center"),
    margin=dict(l=100, r=50, t=80, b=100)
 )
fig_hour.update_xaxes(title_text="Hour of Day (0-23)")
fig_hour.update_yaxes(title_text="Number of Crashes")
fig_hour.show()

**Crash distribution by hour of day**  

Crashes cluster strongly around the two daily commute windows, peaking near 08:00 and again between 16:00–18:00, with a secondary plateau through mid-afternoon. The overnight trough (roughly 02:00–05:00) is pronounced. This volume pattern foreshadows a key finding in Chapter 5.3: rush-hour dominance in *count* does not match the late-night dominance in *severity rate*, revealing a temporal decoupling between how often crashes happen and how bad they are when they do.

In [6]:
### Top 10 Maneuvers at Time of Crash
maneuver_dist = crash_base_basic["MANEUVER"].value_counts().head(10)
fig_maneuver = go.Figure(
    data=[
        go.Bar(
            x=maneuver_dist.values,
            y=maneuver_dist.index,
            text=maneuver_dist.values,
            texttemplate="%{text:.3s}",
            textposition="outside",
            cliponaxis=False,
            orientation="h",
            marker_color=ORANGE,
            opacity=GLOBAL_OPACITY,
        )
    ]
)
_simple_layout(fig_maneuver, "Top 10 Maneuvers at Time of Crash", height=500)
fig_maneuver.update_layout(
    title=dict(x=0.5, xanchor="center"),
    margin=dict(l=275, r=50, t=80, b=80)
 )
fig_maneuver.update_xaxes(title_text="Count")
fig_maneuver.update_yaxes(tickfont=dict(size=12))
fig_maneuver.show()

**Top 10 maneuvers at time of crash**  

Straight-ahead travel is by far the most common maneuver context, unsurprising given that most road distance is travelled in a straight line. Turning maneuvers (left turn, right turn, u-turn) collectively form the next largest group, pointing to intersections and merging points as hotspots for conflict. A small tail of reversing and lane-change maneuvers rounds out the picture. The concentration in a handful of maneuver types suggests that intervention doesn't need to address exotic scenarios; the everyday, high-frequency movements produce most incidents.

In [7]:
### Crash Type Distribution (Top 12)
crash_type_dist = crash_base_basic["FIRST_CRASH_TYPE"].value_counts().head(12)
fig_type = go.Figure(
    data=[
        go.Bar(
            x=crash_type_dist.values,
            y=crash_type_dist.index,
            text=crash_type_dist.values,
            texttemplate="%{text:.3s}",
            textposition="outside",
            cliponaxis=False,
            orientation="h",
            marker_color=ORANGE,
            opacity=GLOBAL_OPACITY,
        )
    ]
)
_simple_layout(fig_type, "Top 12 Types of First Hit of the Crash", height=500)
fig_type.update_layout(
    title=dict(x=0.5, xanchor="center"),
    margin=dict(l=220, r=50, t=80, b=80)
 )
fig_type.update_xaxes(title_text="Count")
fig_type.update_yaxes(tickfont=dict(size=12))
fig_type.show()

**Top 12 first-crash types**  

Rear-end collisions dominate by a wide margin, reflecting the stop-and-go pattern characteristic of dense urban traffic. Turning crashes and angle (broadside) collisions are the next most common types, reinforcing the intersection theme from the maneuver chart. Sideswipe and fixed-object collisions follow. Importantly, the crash types associated with higher severity in the literature — angle, pedestrian/cyclist involvement, and head-on — are present but not the most frequent, underlining the distinction between crash volume and crash severity that anchors this project's framing.

# 4. Genre

This data story uses the **Martini Glass** structure (Segel & Heer, 2010): an author-guided opening funnel that moves from citywide crash patterns to the specific corridors and factors most relevant to policy, then opens into reader-driven exploration via dropdowns and group filters at the figure level.

---

## Visual Narrative Tools (Segel & Heer, Figure 7)

**Visual Structuring** — Every chapter follows the same *intro → visualisation → interpretation* pattern for predictability. Small-multiple grids (the trend chart and both radar sets) display all 15 streets at a shared scale. Linked dropdown menus reuse the same figure body across selections, preserving spatial memory.

**Highlighting** — Severe vs. non-severe crashes are distinguished by a consistent orange/brown colour scheme; trend slopes are red (worsening) or green (improving). Marker size encodes fatal-crash count in the risk scatter. Slope values and rush-hour windows are annotated directly on the figures so readers never need a separate legend lookup.

**Transition Guidance** — Five numbered chapter headings give readers an explicit position in the story. Each figure is framed by a *what-this-shows* paragraph before and a *key-findings* paragraph after.

---

## Narrative Structure Tools (Segel & Heer, Figure 7)

**Ordering** — Chapters are fixed in sequence (overview → spatial → time → causes → synthesis), matching the analytic logic of moving from *what* to *where* to *when* to *why*, with each chapter zooming from citywide to street-level.

**Interactivity** — Dropdown menus filter by road-user group and street set without leaving the figure. Rich hover tooltips expose exact rates and counts. Reader-created annotations are deliberately excluded to preserve narrative clarity.

**Messaging** — The same shrinkage-adjusted severe rate is used throughout so comparisons are always like-for-like. Each chapter ends with a plain-language takeaway. Limitations (e.g. inability to establish causality from traffic-control figures) are noted inline rather than buried in a disclaimer.

# 5. Visualizations - Data Story

## 5.1. Method - Measuring Dangerous Streets

Not all streets are equally dangerous. Counting crashes alone can be misleading because busy roads generate many minor collisions while quieter corridors may carry far higher per-crash risk. Without traffic volume data to normalise by exposure, raw counts are a proxy for volume — not danger.

We define a **severe crash** as any crash with at least one fatal or incapacitating injury. To keep street-level rates statistically stable, we apply **Bayesian shrinkage** (also called empirical Bayes), blending each street's observed severe rate toward the citywide baseline. The shrinkage formula is:

> adjusted\_rate = (severe\_crashes + k × p₀) / (total\_crashes + k)

where **p₀** is the citywide severe rate and **k = 50** is the shrinkage parameter. Setting k = 50 gives the prior a concrete interpretation: a street must accumulate the equivalent of 50 crashes at the citywide rate before its observed severity rate carries more weight than the city baseline. Streets with very few crashes are pulled strongly toward p₀, while streets with hundreds of crashes are barely affected. A data-driven estimate of k was computed initially but produced values close to 50 for the Chicago dataset, and the fixed value was preferred because it is transparent, stable across different data subsets, and gives the reader an intuitive sense of exactly how much regularisation is being applied. Streets with fewer than 30 total crashes are excluded from display entirely.

We also compute a **composite danger score** that weights fatality risk separately from incapacitating injury risk (fatalities are upweighted by a factor of 5), because a street with one death in 30 crashes represents a qualitatively different risk from one with one death in 3,000.

The scatter plot below compares all streets across four dimensions simultaneously:

- **x-axis**: total crash count (exposure proxy)
- **y-axis**: shrinkage-adjusted severe rate
- **marker size**: fatal crash count
- **color**: street type (Avenue, Boulevard, Road, etc.)
- **dropdown**: road-user group filter (All / Driver / Passenger / Pedestrian / Cyclist)

This combined view separates high-volume corridors from streets where crashes are rarer but disproportionately severe — which is the core definition of danger used throughout this project.

In [8]:
def _clean_text(s):
    return s.fillna("Unknown").astype(str).str.strip().replace("", "Unknown")

def _person_group(person_type, is_ped_cyc):
    p = str(person_type).upper()
    ped_cyc = bool(is_ped_cyc)

    if ped_cyc:
        if any(k in p for k in ["CYCL", "BICYC", "PEDAL", "BIKE"]):
            return "Cyclist"
        return "Pedestrian"

    if "PASSENGER" in p:
        return "Passenger"

    return "Driver"

def _pick_primary(gs):
    priority = ["Pedestrian", "Cyclist", "Passenger", "Driver"]
    for g in priority:
        if g in gs:
            return g
    return "Driver"


# Unique-crash table for all crash-level summaries
crash_base = df.drop_duplicates(subset=["CRASH_RECORD_ID"]).copy()
crash_base = crash_base.dropna(subset=["STREET_NAME"]).copy()

crash_base["is_severe_crash"] = (
    (crash_base["INJURIES_FATAL"].fillna(0) > 0)
    | (crash_base["INJURIES_INCAPACITATING"].fillna(0) > 0)
)

crash_base["is_deadly_crash"] = (crash_base["INJURIES_FATAL"].fillna(0) > 0)

crash_base["CRASH_YEAR"] = pd.to_numeric(crash_base["CRASH_YEAR"], errors="coerce")
crash_base["CRASH_HOUR"] = pd.to_numeric(crash_base["CRASH_HOUR"], errors="coerce").fillna(0).clip(0, 23).astype(int)

# Build one "primary affected group" per crash for stacked composition over years
person_cols = ["CRASH_RECORD_ID", "PERSON_TYPE", "IS_PEDESTRIAN_CYCLIST"]
person_view = df[person_cols].copy()
person_view["PERSON_TYPE"] = person_view["PERSON_TYPE"].fillna("Unknown")
person_view["person_group"] = [
    _person_group(pt, pc)
    for pt, pc in zip(person_view["PERSON_TYPE"], person_view["IS_PEDESTRIAN_CYCLIST"])
]

group_sets = (
    person_view.groupby("CRASH_RECORD_ID")["person_group"]
    .apply(lambda s: set(s.dropna().tolist()))
    .reset_index(name="group_set")
)

group_sets["primary_group"] = group_sets["group_set"].apply(_pick_primary)

crash_base = crash_base.merge(
    group_sets[["CRASH_RECORD_ID", "primary_group"]],
    on="CRASH_RECORD_ID",
    how="left",
)
crash_base["primary_group"] = crash_base["primary_group"].fillna("Driver")

# --- count people types per street ---
person_view = person_view.merge(
    crash_base[["CRASH_RECORD_ID", "STREET_NAME"]],
    on="CRASH_RECORD_ID",
    how="left"
)

people_counts = (
    person_view.groupby(["STREET_NAME", "person_group"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# --- number of people per crash ---
people_per_crash = (
    df.groupby("CRASH_RECORD_ID")
    .size()
    .reset_index(name="people_in_crash")
)
crash_base = crash_base.merge(
    people_per_crash,
    on="CRASH_RECORD_ID",
    how="left"
)

# Top-15 dangerous streets by shrunken severe-crash risk per 100 crashes
street_risk = (
    crash_base.groupby("STREET_NAME", as_index=False)
    .agg(
        total_crashes=("CRASH_RECORD_ID", "nunique"),
        severe_crashes=("is_severe_crash", "sum"),
        deadly_crashes=("is_deadly_crash", "sum"),
        avg_people_per_crash=("people_in_crash", "mean"),
    )
)

# ── Compute m and p0 HERE, before any shrinkage block ──────────────────────
p0 = street_risk["severe_crashes"].sum() / street_risk["total_crashes"].sum()
# m = int(p0 * (1 - p0) / street_risk["severe_crashes"].var() * street_risk["total_crashes"].mean())
m = 50
street_risk["avg_people_per_crash"] = street_risk["avg_people_per_crash"].round(2)

# ── Street shrinkage (p0 and m already computed above) ─────────────────────
street_risk["risk_per_100"] = 100 * (
    (street_risk["severe_crashes"] + m * p0) / (street_risk["total_crashes"] + m)
)

p0_1 = street_risk["deadly_crashes"].sum() / street_risk["total_crashes"].sum()
street_risk["death_risk_per_100"] = 100 * (
    (street_risk["deadly_crashes"] + m * p0_1) /
    (street_risk["total_crashes"] + m)
)
# 2) Add a minimum crash threshold AFTER shrinkage for display purposes only
# (don't filter before — shrinkage handles low-n streets, but streets with
# n=3 crashes are noise even after shrinkage)
street_risk = street_risk[street_risk["total_crashes"] >= 30]

# 3) Consider a COMPOSITE score rather than just severe rate:
# weight severe + deadly separately, since a street with 1 death in 10 
# crashes is very different from 1 death in 500
street_risk["composite_score"] = (
    0.5 * street_risk["risk_per_100"] +
    0.5 * street_risk["death_risk_per_100"] * 5  # upweight fatalities
)

street_risk = street_risk.merge(
    people_counts,
    on="STREET_NAME",
    how="left"
)

# fill missing columns if some groups don't exist
for col in ["Pedestrian", "Cyclist", "Passenger", "Driver"]:
    if col not in street_risk:
        street_risk[col] = 0

street_risk[["Pedestrian", "Cyclist", "Passenger", "Driver"]] = (
    street_risk[["Pedestrian", "Cyclist", "Passenger", "Driver"]].fillna(0)
)

street_risk = street_risk.merge(
    crash_base[["CRASH_RECORD_ID", "STREET_NAME", "ZIPCODE"]]
    .drop_duplicates(subset=["STREET_NAME"]),
    on="STREET_NAME",
    how="left"
)


# -------------------------------
# Top 15 streets by total crashes
# -------------------------------
street_total = (
    crash_base.groupby("STREET_NAME", as_index=False)
    .agg(
        total_crashes=("CRASH_RECORD_ID", "nunique"),
        severe_crashes=("is_severe_crash", "sum"),
        deadly_crashes=("is_deadly_crash", "sum"),
    )
)

street_risk["severe_prob"] = (
    street_risk["severe_crashes"] / street_risk["total_crashes"]
)

street_risk["death_prob"] = (
    street_risk["deadly_crashes"] / street_risk["total_crashes"]
)

street_risk["pedestrian_risk"] = (
    street_risk["deadly_crashes"] / (street_risk["Pedestrian"] + 1e-9)
)

street_risk["cyclist_risk"] = (
    street_risk["deadly_crashes"] / (street_risk["Cyclist"] + 1e-9)
)

# street_total = street_total.sort_values(
#     "total_crashes",
#     ascending=False
# ).head(15).copy()

# top15_streets = street_total["STREET_NAME"].tolist()

In [9]:
### Risk vs. Volume per street, and person group
GROUPS = ["All", "Driver", "Passenger", "Pedestrian", "Cyclist"]
color_map = {
    "AVE": BLUE, "PL": YELLOW, "ST": GREEN,
    "DR": BROWN, "RD": PURPLE, "BLVD": ORANGE,
}

def _street_type_from_name(name):
    import re
    s = str(name).upper()
    patterns = {
        "AVE":  ["AVE", "AVENUE"],
        "BLVD": ["BLVD", "BOULEVARD"],
        "DR":   ["DR",  "DRIVE"],
        "RD":   ["RD",  "ROAD"],
        "ST":   ["ST",  "STREET"],
        "PL":   ["PL",  "PLACE"],
    }
    for key, tokens in patterns.items():
        for token in tokens:
            if re.search(rf"\b{token}\b", s):
                return key
    return "OTHER"


def _compute_street_risk(df_sub):
    sr = (
        df_sub.groupby("STREET_NAME", as_index=False)
        .agg(
            total_crashes=("CRASH_RECORD_ID", "nunique"),
            severe_crashes=("is_severe_crash", "sum"),
            deadly_crashes=("is_deadly_crash", "sum"),
        )
    )
    sr = sr[sr["total_crashes"] >= 30].copy()

    p0_s  = sr["severe_crashes"].sum() / sr["total_crashes"].sum()
    var_s = sr["severe_crashes"].var()
    m_s   = int(p0_s * (1 - p0_s) / var_s * sr["total_crashes"].mean()) if var_s > 0 else 1
    sr["severe_rate"] = 100 * (
        (sr["severe_crashes"] + m_s * p0_s) / (sr["total_crashes"] + m_s)
    )

    sr["STREET_TYPE"] = sr["STREET_NAME"].apply(_street_type_from_name)
    sr["COLOR"]       = sr["STREET_TYPE"].map(color_map).fillna(GRAY)
    sr["GROUP"]       = sr["STREET_TYPE"].where(
        sr["STREET_TYPE"].isin(color_map.keys()), "OTHER"
    )

    size_raw = sr["deadly_crashes"]
    sr["marker_size"] = (
        (size_raw - size_raw.min()) /
        (size_raw.max() - size_raw.min() + 1e-9) * 25 + 6
    )
    sr["show_label"] = (
        (sr["total_crashes"] > 15_000) |
        (sr["severe_rate"] >= sr["severe_rate"].quantile(0.95))
    )
    return sr


group_sr = {}
for grp in GROUPS:
    if grp == "All":
        sub = crash_base.copy()
    else:
        ids = crash_base.loc[crash_base["primary_group"] == grp, "CRASH_RECORD_ID"]
        sub = crash_base[crash_base["CRASH_RECORD_ID"].isin(ids)].copy()
    group_sr[grp] = _compute_street_risk(sub)


# ── Build figure — one set of traces per GROUP, repeated for each GROUPS tab ──
#
# Strategy: add len(all_street_types) traces per person-group.
# Only the traces belonging to the active person-group are visible at a time.
# Each dropdown button flips visibility with a single restyle call (no Python
# callback required — everything lives inside the figure JSON).

sr_default         = group_sr["All"]
all_street_types   = sorted(sr_default["GROUP"].unique())   # e.g. AVE, BLVD, …
n_street_types     = len(all_street_types)

fig = go.Figure()

for grp in GROUPS:
    sr      = group_sr[grp]
    visible = (grp == "All")          # only "All" traces shown initially

    for stype in all_street_types:
        df = sr[sr["GROUP"] == stype]
        fig.add_trace(go.Scatter(
            x=df["total_crashes"],
            y=df["severe_rate"],
            mode="markers",
            name=stype,
            visible=visible,
            showlegend=(grp == "All"),   # legend entries only from the first set
            marker=dict(
                size=df["marker_size"],
                color=df["COLOR"],
                opacity=GLOBAL_OPACITY,
                line=dict(width=0.5, color="#333"),
            ),
            text=df["STREET_NAME"],
            customdata=df[["deadly_crashes", "severe_crashes", "severe_rate"]].values,
            hovertemplate=(
                "<b>%{text}</b><br>"
                "Total crashes: %{x}<br>"
                "Fatal crashes: %{customdata[0]}<br>"
                "Severe crashes: %{customdata[1]}<br>"
                "Severe rate: %{customdata[2]:.2f}%"
                "<extra></extra>"
            ),
        ))

# total traces = len(GROUPS) * n_street_types
# for group index g, its traces occupy slots [g*n : g*n + n]

def _visibility_mask(active_idx):
    """Return a list of True/False for every trace."""
    mask = []
    for g in range(len(GROUPS)):
        mask.extend([g == active_idx] * n_street_types)
    return mask

# ── Axis ranges per group (pre-computed so the dropdown can set them too) ─────
def _axis_ranges(grp):
    sr    = group_sr[grp]
    all_x = sr["total_crashes"].tolist()
    all_y = sr["severe_rate"].tolist()
    x_pad = (max(all_x) - min(all_x)) * 0.08 or 500
    y_pad = (max(all_y) - min(all_y)) * 0.12 or 1
    return (
        [max(0, min(all_x) - x_pad), max(all_x) + x_pad],
        [max(0, min(all_y) - y_pad), max(all_y) + y_pad],
    )

# ── Dropdown buttons ──────────────────────────────────────────────────────────
buttons = []
for i, grp in enumerate(GROUPS):
    xr, yr = _axis_ranges(grp)
    buttons.append(dict(
        label=grp,
        method="update",          # updates both traces AND layout in one call
        args=[
            {"visible": _visibility_mask(i)},
            {
                "xaxis.range":     xr,
                "xaxis.autorange": False,
                "yaxis.range":     yr,
                "yaxis.autorange": False,
            },
        ],
    ))

# ── Layout ────────────────────────────────────────────────────────────────────
xr0, yr0 = _axis_ranges("All")

_simple_layout(fig, title="Street Risk vs. Crash Volume", height=650)

fig.update_layout(
    title=dict(y=0.97, x=0.425, xanchor="left"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    updatemenus=[dict(
        type="buttons",
        direction="right",
        active=0,
        x=0.5, xanchor="center",
        y=1.13, yanchor="top",
        buttons=buttons,
        pad={"r": 6, "t": 6, "b": 6, "l": 6},
        bgcolor="#f9fafb",
        bordercolor="#d0d5dd",
        borderwidth=2,
        font=dict(family=FONT, size=14, color="#344054"),
        showactive=True,
    )],
    legend=dict(
        itemsizing="constant",
        title="Street Type",
        title_font=dict(color=LABEL_COLOR, family=FONT, size=14),
        font=dict(size=14, family=FONT, color=LABEL_COLOR),
        orientation="h",
        yanchor="top", y=-0.18,
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.7)",
    ),
    margin=dict(l=90, r=40, t=120, b=120),
    hoverlabel=dict(
        font=dict(family=FONT, size=13, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
    xaxis=dict(
        title_text="Total Crashes",
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        zeroline=True, zerolinecolor="#D4D4D4", zerolinewidth=2,
        range=xr0, autorange=False, nticks=10,
        title_font=dict(color=LABEL_COLOR, family=FONT, size=18),
        title_standoff=25,
        tickfont=dict(color=LABEL_COLOR, family=FONT, size=14),
    ),
    yaxis=dict(
        title_text="Severe Rate (%)",
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        zeroline=True, zerolinecolor="#D4D4D4", zerolinewidth=2,
        range=yr0, autorange=False, nticks=8,
        title_font=dict(color=LABEL_COLOR, family=FONT, size=18),
        title_standoff=20,
        tickfont=dict(color=LABEL_COLOR, family=FONT, size=14),
    ),
    annotations=[dict(
        text="Marker size = number of fatal crashes",
        xref="paper", yref="paper",
        x=0.97, y=0.97, xanchor="right", yanchor="top",
        showarrow=False,
        font=dict(size=14, color=LABEL_COLOR, family=FONT),
        align="right", bgcolor="white",
        bordercolor=LABEL_COLOR, borderwidth=1,
    )],
)

fig.show()

### Visualization: Street Risk vs. Crash Volume — Interactive Scatter Plot

**What the plot shows:** Every named street in Chicago with at least 30 recorded crashes appears as one dot. The horizontal axis is total crash count (a proxy for exposure), the vertical axis is the shrinkage-adjusted severe crash rate, marker size encodes fatal crash count, and colour encodes street type. An interactive group selector filters the view to crashes involving only a specific road-user group.

**Why this visualization is right for the story:** The opening analytical question is "which streets are truly dangerous?" — and the answer depends on separating *volume* from *severity*. A simple bar chart of crash counts would highlight the busiest streets, not the most dangerous ones. A scatter plot with the two dimensions on separate axes makes that distinction visually immediate: streets in the upper-left quadrant are rare-crash/high-severity (dangerous per event), while streets in the lower-right are high-volume/low-severity (dangerous by accumulation). Marker size adds a third dimension — fatality count — without requiring a separate chart. The group selector turns one figure into five, allowing comparison across road-user types without multiplying chart count.

Reading the plot:

* Each point is a street with at least 30 crashes (to avoid very small samples).
* Higher points mean a higher severe-crash rate; farther right means more crashes.
* Larger markers indicate more fatal crashes, which highlights streets where outcomes are worst.
* Colors show street type, and the dropdown switches the road-user group.

Citywide patterns:

* Most streets cluster at low volumes and under about 4% severe rate, showing that street type alone does not explain severity.
* A small set of streets are clear outliers with unusually high severe rates despite modest volumes.
* Major arterials (for example, Western Ave and Pulaski Rd) dominate total crash counts but sit at lower severe rates, which still yields many severe outcomes in absolute numbers.

Differences by road user group:

* Drivers and passengers largely mirror the citywide pattern.
* Pedestrian crashes are less frequent but far more severe on several corridors, which pushes some streets high on the severity axis.
* Cyclist crashes cluster at lower volumes, with a few corridors standing out as both busy and risky.

Top 15 streets used in the rest of the analysis:

* Western Avenue
* Pulaski Road
* Cicero Avenue
* Russell Drive
* 105th Place
* Vermont Street
* Lake Shore Drive (northbound)
* Ashland Avenue
* Halsted Street
* Milwaukee Avenue
* Taylor Street
* 59th Street
* Dr Martin Luther King Jr Drive
* 71st Street
* Damen Avenue

These streets were selected by visual inspection of the risk–volume scatter plot rather than by a data-driven threshold. For each road user group, the streets that appeared as clear outliers — standing out along the severity axis, the volume axis, or isolated from the main cluster were included. This approach treats the scatter plot as the analytical tool it was designed to be: a way to make the outliers visible so that they can be named and studied. A strict data-driven ranking by severity rate alone would have selected very different streets (many with fewer than 100 total crashes), while ranking by crash count alone would have selected only the busiest arterials. Combining both dimensions in the scatter and reading the result visually is what gives the top-15 list its analytical meaning.

These streets anchor the rest of the analysis, where we look at timing and contributing risk factors in more detail.

Why this visualization works: the scatter separates exposure (volume) from severity, marker size adds fatality context, and color encodes street type without clutter.

In [10]:
top_streets = [
    "WESTERN AVE", "PULASKI RD", "CICERO AVE", "RUSSELL DR", "105TH PL", "VERMONT ST", 
    "LAKE SHORE DR NB", "ASHLAND AVE", "HALSTED ST",
    "MILWAUKEE AVE", 
    "TAYLOR ST", "59TH ST", "DR MARTIN LUTHER KING JR DR", "71ST ST", "DAMEN AVE"
]

In [11]:
### Time series of severe crash rate per street
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as sp_stats

# --- per-street per-year aggregation (shrunk severe rate) ---
m_trend = 50
p0_trend = crash_base["is_severe_crash"].mean()

yr_street = (
    crash_base.groupby(["STREET_NAME", "CRASH_YEAR"], as_index=False)
    .agg(
        total=("CRASH_RECORD_ID", "nunique"),
        severe=("is_severe_crash", "sum"),
    )
)
yr_street = yr_street.dropna(subset=["CRASH_YEAR"])
yr_street["CRASH_YEAR"] = yr_street["CRASH_YEAR"].astype(int)
yr_street["shrunk_rate"] = (
    (yr_street["severe"] + m_trend * p0_trend) /
    (yr_street["total"] + m_trend)
) * 100

top15 = top_streets  

cols = 5
rows = 3
fig_trend = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=top15,
    horizontal_spacing=0.06,
    vertical_spacing=0.14,
)

for idx, street in enumerate(top15):
    row = idx // cols + 1
    col = idx % cols + 1

    sdf = yr_street[yr_street["STREET_NAME"] == street].sort_values("CRASH_YEAR")
    if len(sdf) < 2:
        continue

    x = sdf["CRASH_YEAR"].values.astype(float)
    y = sdf["shrunk_rate"].values

    slope, intercept, *_ = sp_stats.linregress(x, y)
    trend_y = slope * x + intercept

    # colour by slope direction
    trend_color = "#D65F5F" if slope > 0 else "#54A24B"  # red=worse, green=better

    # actual rate line
    fig_trend.add_trace(
        go.Scatter(
            x=sdf["CRASH_YEAR"], y=y,
            mode="lines+markers",
            line=dict(color=ORANGE, width=1.8),
            marker=dict(size=5, color=BROWN),
            showlegend=False,
            hovertemplate="Year: %{x}<br>Severe rate: %{y:.1f}%<extra></extra>",
        ),
        row=row, col=col,
    )

    # OLS trend line
    fig_trend.add_trace(
        go.Scatter(
            x=sdf["CRASH_YEAR"], y=trend_y,
            mode="lines",
            line=dict(color=trend_color, width=2, dash="dashdot"),
            showlegend=False,
            hovertemplate=f"Trend slope: {slope:+.2f}%/yr<extra></extra>",
        ),
        row=row, col=col,
    )

    # slope annotation inside subplot
    fig_trend.add_annotation(
        text=f"{slope:+.2f}%/yr",
        x=x[-1], y=trend_y[-1],
        xref=f"x{idx+1}", yref=f"y{idx+1}",
        showarrow=False,
        font=dict(size=9, color=trend_color),
        xanchor="right",
    )

# legend proxies
fig_trend.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#D65F5F", width=2, dash="dashdot"), name="Trend (improving)", showlegend=True))
fig_trend.add_trace(go.Scatter(x=[None], y=[None], mode="lines",
    line=dict(color="#54A24B", width=2, dash="dashdot"), name="Trend (worsening)", showlegend=True))

_simple_layout(fig_trend, "", height=680)
fig_trend.update_layout(
    title=dict(text="<b>Year-over-Year Severe Crash Rate per Street</b>", x=0.5, xanchor="center"),
    legend=dict(orientation="h", yanchor="top", y=-0.08, xanchor="center", x=0.5, title=None, font=dict(size=16, color=LABEL_COLOR)),
    margin=dict(l=50, r=30, t=100, b=80),
)
fig_trend.update_annotations(font=dict(size=11, color="#262626"))
fig_trend.update_yaxes(ticksuffix="%")
fig_trend.show()

### Visualization: Severe Crash Rate Trends — Small Multiples

**What the plot shows:** A 3 × 5 grid of small panels, one per top-15 street, each showing the year-by-year shrinkage-adjusted severe crash rate from 2015 to the present. The orange line is the observed annual rate; a dashed OLS trend line summarises the direction. Slope values (annotated directly: e.g. +0.12 %/yr) are coloured red if worsening and green if improving.

**Why this visualization:** After establishing *which* streets are dangerous (the scatter in Chapter 1), the natural next question is: are they getting better or worse over time? Small multiples are the canonical answer: 15 panels on one canvas allow the reader to scan and compare direction across all corridors simultaneously, without the visual tangle of 15 overlapping lines on a single chart. The shared y-axis range makes severity magnitudes comparable across panels. The OLS trend line summarises a noisy time series into a single, directional signal, while the raw orange line preserves the year-to-year variation that context like COVID-19 or policy changes can explain.

Each small panel shows the yearly shrinkage-adjusted severe rate for one of the top 15 streets. The solid line is the observed rate, and the dashed line is a linear trend that summarizes direction over time.

* Green slopes indicate improving safety (lower severity over time).
* Red slopes indicate worsening safety (higher severity over time).
* Flat slopes suggest stability rather than change.

The overall picture is mixed rather than uniform. Some corridors show slow improvement (e.g. Lake Shore DR NB, Ashland AVE and 71ST ST), others drift upward, and several remain roughly flat. This reinforces the street-level framing of the project: citywide averages hide local changes, and the most dangerous streets do not move in lockstep.

### Visualization: Severe Crash Composition Over Time — Stacked Bars

**What the plot shows:** Two stacked bar charts (one for the top-15 streets, one for all streets citywide) showing the annual count of severe crashes broken down by primary road-user group (Driver, Passenger, Pedestrian, Cyclist).

**Why this visualization:** The trend lines above show *how much* severity has changed over time. The stacked bars show *who* is bearing that severity. A stacked bar chart is the natural encoding for part-to-whole relationships over time: it simultaneously conveys the total (bar height) and the composition (colour splits). A 100% stacked bar would show only proportional shifts, hiding the absolute change in counts and  separate lines per group would make the total invisible. The side-by-side comparison between top streets and all streets directly tests whether the most dangerous corridors have a distinctive user-group mix, or simply concentrate the same city-wide mix at higher intensity.

In [12]:
### Decomposition of severe crash composition over years, for top streets vs. all streets
groups = ["Pedestrian", "Cyclist", "Passenger", "Driver"]
group_colors = {
    "Pedestrian": ORANGE,
    "Cyclist":    PURPLE,
    "Passenger":  GREEN,
    "Driver":     BLUE,
}

# --- Data prep ---
decomp_df = crash_base[crash_base["is_severe_crash"] == True].copy()
decomp_df = decomp_df.dropna(subset=["CRASH_YEAR"])
decomp_df["CRASH_YEAR"] = decomp_df["CRASH_YEAR"].astype(int)

top_pivot = (
    decomp_df[decomp_df["STREET_NAME"].isin(top_streets)]
    .groupby(["CRASH_YEAR", "primary_group"])["CRASH_RECORD_ID"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=groups, fill_value=0)
    .reset_index()
)

all_pivot = (
    decomp_df
    .groupby(["CRASH_YEAR", "primary_group"])["CRASH_RECORD_ID"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=groups, fill_value=0)
    .reset_index()
)

# --- Figure ---
fig_decomp = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>Severe Crash Composition — Top Streets</b>",
        "<b>Severe Crash Composition — All Streets</b>",
    ),
    horizontal_spacing=0.10,
)

for g in groups:
    # Left subplot — top streets
    fig_decomp.add_trace(go.Bar(
        x=top_pivot["CRASH_YEAR"],
        y=top_pivot[g],
        name=g,
        marker_color=group_colors[g],
        opacity=GLOBAL_OPACITY,
        legendgroup=g,
        showlegend=True,
        hovertemplate=f"<b>{g}</b><br>Year: %{{x}}<br>Severe crashes: %{{y}}<extra></extra>",
    ), row=1, col=1)

    # Right subplot — all streets (reuse same legend entry)
    fig_decomp.add_trace(go.Bar(
        x=all_pivot["CRASH_YEAR"],
        y=all_pivot[g],
        name=g,
        marker_color=group_colors[g],
        opacity=GLOBAL_OPACITY,
        legendgroup=g,
        showlegend=False,
        hovertemplate=f"<b>{g}</b><br>Year: %{{x}}<br>Severe crashes: %{{y}}<extra></extra>",
    ), row=1, col=2)

_simple_layout(fig_decomp, "", height=520)

fig_decomp.update_layout(
    barmode="stack",
    legend=dict(
        orientation="h", yanchor="top", y=-0.18,
        xanchor="center", x=0.5,
        font=dict(size=14, family=FONT, color=LABEL_COLOR),
    ),
    margin=dict(l=60, r=60, t=90, b=100),
    hoverlabel=dict(
        font=dict(family=FONT, size=13, color=LABEL_COLOR),
        bgcolor="white",
        bordercolor=GRAY,
    ),
)

# Apply axis styling to both subplots
for col in (1, 2):
    fig_decomp.update_xaxes(
        title_text="Year", dtick=1,
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color=LABEL_COLOR, size=12, family=FONT),
        title_font=dict(color=LABEL_COLOR, size=16, family=FONT),
        row=1, col=col,
    )
    fig_decomp.update_yaxes(
        title_text="Severe crashes",
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        title_font=dict(color=LABEL_COLOR, size=16, family=FONT),
        tickfont=dict(color=LABEL_COLOR, size=12, family=FONT),
        row=1, col=col,
    )

fig_decomp.show()

This stacked bar chart aggregates severe crashes on the top streets by primary road-user group over time. It answers a simple question: who is most often involved when severe crashes happen on the city's most dangerous corridors?

* Passengers account for the largest share in most years.
* Drivers are typically the second largest share.
* Pedestrians and cyclists make up a smaller but persistent portion.

The mix is not constant year to year. Even when total severe crashes fluctuate, vulnerable users remain visible in the composition, which is why later sections keep user groups separate instead of treating all crashes as the same.

Importantly, the distribution of user types for severe crashes across all streets and across the top streets is very similar, which suggests the top corridors concentrate the same user mix rather than a completely different one.

## 5.2 Where Crashes Happen

### Visualization: Chicago Street Crashes 2022-2026 (Scattermap)

**What the plot shows:** A single-panel interactive map with a year slider (2022-2026). Each point is a crash; color encodes severity: no injury, at least one incapacitating injury, or at least one fatality. Toggles for All streets and Top Streets make it possible to filter the map focusing only on the top streets.

**Why this visualization:** This chapter asks whether crash risk has a geographic pattern. The narrative moves from citywide statistics to named streets, but it skips the spatial step of showing where those streets are. The map makes the story concrete for readers familiar with Chicago and shows whether risk is tied to location (for example, proximity to highways or dense areas) or to specific streets. 

In [13]:
### Data loading and preprocessing for the map viz
import numpy as np
from shapely.geometry import LineString, MultiLineString
import re, json

CRASHES_CSV     = "Traffic_Crashes_-_Crashes_20260414.csv"
STREET_INFO_CSV = "street_geojson_info.csv"
STREETS_GEOJSON = "transportation_20260416.geojson"

YEAR_MIN, YEAR_MAX = 2022, 2026
BAYESIAN_M = 50
POINT_SAMPLE = 0.12
MAX_LINES = 2000
SIMPLIFY_TOL = 0.0002
NUM_BINS = 4

TOP_STREETS = {
    "WESTERN AVE","PULASKI RD","CICERO AVE","RUSSELL DR","105TH PL",
    "VERMONT ST","LAKE SHORE DR NB","ASHLAND AVE","HALSTED ST",
    "MILWAUKEE AVE","TAYLOR ST","59TH ST","DR MARTIN LUTHER KING JR DR",
    "71ST ST","DAMEN AVE"
}

POINT_COLORS = {
    "No injury": "rgba(148,163,184,0.75)",
    "Incapacitating": "rgba(245,158,11,0.85)",
    "Fatal": "rgba(239,68,68,0.9)",
}

def normalize_street_name(value):
    if pd.isna(value):
        return None
    s = str(value).upper().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^A-Z0-9 ]", "", s)
    return s or None

def geometry_groups(geometry):
    gtype  = (geometry or {}).get("type")
    coords = (geometry or {}).get("coordinates", [])
    if gtype == "LineString":
        return [coords]
    if gtype == "MultiLineString":
        return coords
    return []

crashes = pd.read_csv(CRASHES_CSV)
streets = pd.read_csv(STREET_INFO_CSV)

date_col = "CRASH_DATE" if "CRASH_DATE" in crashes.columns else "CRASH_DATE_x"
crashes["CRASH_DATE_PARSED"] = pd.to_datetime(crashes[date_col], errors="coerce")
crashes = crashes[crashes["CRASH_DATE_PARSED"].notna()].copy()
crashes["CRASH_YEAR"] = crashes["CRASH_DATE_PARSED"].dt.year
crashes = crashes[crashes["CRASH_YEAR"].between(YEAR_MIN, YEAR_MAX)].copy()

street_col = next((c for c in ["STREET_NAME","ON_STREET_NAME","STREET_NAM","STREET"] if c in crashes.columns), None)
if not street_col:
    raise KeyError("No street column found in crashes.")

crashes = crashes.drop_duplicates(subset=["CRASH_RECORD_ID"]).copy()
crashes["IS_SEVERE"] = (
    (crashes["INJURIES_FATAL"].fillna(0) > 0) |
    (crashes["INJURIES_INCAPACITATING"].fillna(0) > 0)
)
crashes["IS_FATAL"] = crashes["INJURIES_FATAL"].fillna(0) > 0

def injury_level(row):
    if row["INJURIES_FATAL"] > 0:
        return "Fatal"
    if row["INJURIES_INCAPACITATING"] > 0:
        return "Incapacitating"
    return "No injury"

crashes["INJURY_LEVEL"] = crashes.apply(injury_level, axis=1)
crashes["STREET_KEY"] = crashes[street_col].map(normalize_street_name)

with open(STREETS_GEOJSON, "r", encoding="utf-8") as f:
    gj = json.load(f)

rows = []
for feat in gj.get("features", []):
    props = feat.get("properties", {}) or {}
    raw_name = None
    for k in ["STREET_NAME","STREET_NAM","STREETNAME","FULL_STREET","FULLNAME","ST_NAME","STREET","RD_NAME","NAME"]:
        if k in props and props[k]:
            raw_name = props[k]
            break
    key = normalize_street_name(raw_name)
    if not key:
        continue
    rows.append((key, feat.get("geometry", {})))

base_geom_df = pd.DataFrame(rows, columns=["STREET_KEY","GEOM"])

years = sorted(crashes["CRASH_YEAR"].dropna().unique().tolist())
if not years:
    raise ValueError("No years found in CRASH_YEAR.")
init_year = max(years)

/var/folders/fm/5lbdpfj928d6sqhy_19cp6ww0000gn/T/ipykernel_61357/677406748.py:47: DtypeWarning:

Columns (0: LANE_CNT) have mixed types. Specify dtype option on import or set low_memory=False.



In [14]:
### Street Crash Map with toggle for All Streets vs Top Streets (severity colors)
fig = go.Figure()
trace_meta = []

bin_colors = ["rgba(200,200,200,0.6)","rgba(252,186,3,0.8)","rgba(252,130,87,0.85)","rgba(217,72,1,0.9)"]

try:
    top_set = set(TOP_STREETS)
except NameError:
    top_set = set(top_streets)

street_sets = {
    "All streets": None,
    "Top streets": top_set,
}

init_street_label = "All streets"

for street_label, street_set in street_sets.items():
    for yr in years:
        crashes_y = crashes[crashes["CRASH_YEAR"] == yr].copy()
        if street_set is not None:
            crashes_y = crashes_y[crashes_y["STREET_KEY"].isin(street_set)]
        if crashes_y.empty:
            continue

        street_risk = (
            crashes_y.groupby("STREET_KEY", dropna=True)
            .agg(total=("CRASH_RECORD_ID","nunique"), severe=("IS_SEVERE","sum"))
            .reset_index()
        )
        p0 = street_risk["severe"].sum() / street_risk["total"].sum()
        m = BAYESIAN_M
        street_risk["risk_per_100"] = 100 * (street_risk["severe"] + m * p0) / (street_risk["total"] + m)

        geom_df = base_geom_df.merge(
            street_risk[["STREET_KEY","risk_per_100"]], on="STREET_KEY", how="left"
        )
        geom_df = geom_df.dropna(subset=["risk_per_100"]).copy()
        geom_df = geom_df.sort_values("risk_per_100", ascending=False).head(MAX_LINES)

        if geom_df.empty or geom_df["risk_per_100"].dropna().empty:
            bins = np.linspace(0, 1, NUM_BINS + 1)
            geom_df["bin"] = 0
        else:
            bins = np.quantile(geom_df["risk_per_100"], np.linspace(0, 1, NUM_BINS + 1))
            geom_df["bin"] = np.clip(
                np.digitize(geom_df["risk_per_100"], bins, right=True) - 1,
                0, NUM_BINS - 1,
            )

        for b in range(NUM_BINS):
            seg_lons, seg_lats = [], []
            sub = geom_df[geom_df["bin"] == b]
            for geom in sub["GEOM"]:
                for coords in geometry_groups(geom):
                    try:
                        ls = LineString(coords).simplify(SIMPLIFY_TOL)
                    except Exception:
                        continue
                    if isinstance(ls, MultiLineString):
                        lines = list(ls)
                    else:
                        lines = [ls]
                    for line in lines:
                        xs, ys = line.xy
                        seg_lons.extend(xs); seg_lats.extend(ys)
                        seg_lons.append(None); seg_lats.append(None)
            if seg_lons:
                fig.add_trace(go.Scattermap(
                    lon=seg_lons, lat=seg_lats, mode="lines",
                    line=dict(color=bin_colors[b], width=1.6),
                    hoverinfo="skip", showlegend=False,
                    visible=(street_label == init_street_label and yr == init_year),
                ))
                trace_meta.append((street_label, yr))

        pts = crashes_y.dropna(subset=["LATITUDE","LONGITUDE"]).copy()
        pts = pts.sample(frac=POINT_SAMPLE, random_state=7)

        for level, color in POINT_COLORS.items():
            sub = pts[pts["INJURY_LEVEL"] == level]
            if sub.empty:
                continue
            fig.add_trace(go.Scattermap(
                lon=sub["LONGITUDE"], lat=sub["LATITUDE"], mode="markers",
                marker=dict(size=5, color=color),
                showlegend=False,
                visible=(street_label == init_street_label and yr == init_year),
                hovertemplate="Severity: " + level + "<extra></extra>",
            ))
            trace_meta.append((street_label, yr))

def make_steps(street_label):
    steps = []
    for yr in years:
        vis = [(m[0] == street_label and m[1] == yr) for m in trace_meta]
        steps.append(dict(method="update", args=[{"visible": vis}], label=str(yr)))
    return steps

legend_lines = ["<br><b>Crash severity</b>"]
for level, color in POINT_COLORS.items():
    legend_lines.append(f'<span style="color:{color}">●</span>  {level}')

slider_all = dict(
    active=years.index(init_year),
    currentvalue=dict(prefix="Year: "),
    pad=dict(t=6),
    x=0.5, y=0.0,
    xanchor="center", yanchor="top",
    steps=make_steps("All streets"),
)

slider_top = dict(
    active=years.index(init_year),
    currentvalue=dict(prefix="Year: "),
    pad=dict(t=6),
    x=0.5, y=0.0,
    xanchor="center", yanchor="top",
    steps=make_steps("Top streets"),
)

buttons = [
    dict(
        label="All streets",
        method="update",
        args=[
            {"visible": [(m[0] == "All streets" and m[1] == init_year) for m in trace_meta]},
            {"sliders": [slider_all]},
        ],
    ),
    dict(
        label="Top streets",
        method="update",
        args=[
            {"visible": [(m[0] == "Top streets" and m[1] == init_year) for m in trace_meta]},
            {"sliders": [slider_top]},
        ],
    ),
]

_simple_layout(fig, title="Chicago Street Crashes between 2022-2026", height=700)

fig.update_layout(
    title=dict(y=0.97, x=0.5, xanchor="center"),
    map=dict(style="carto-positron", center=dict(lat=41.8781, lon=-87.6298), zoom=11),
    margin=dict(l=20, r=20, t=110, b=110),
    height=800,
    showlegend=False,
    annotations=[dict(
        x=0.01, y=0.97,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        text="<br>".join(legend_lines),
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.80)",
        bordercolor="rgba(0,0,0,0.15)",
        borderwidth=1,
        borderpad=6,
        font=dict(size=12, family="Arial"),
    )],
    sliders=[slider_all],
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5, y=1.03,
        xanchor="center", yanchor="bottom",
        buttons=buttons,
        pad=dict(t=8, r=10),
    )],
    uirevision="keep",
)

fig.show()

**Key patterns:**
* Crashes are concentrated downtown and along major arterial roads where traffic volume is highest.
* Severe and fatal crashes appear across the city, including residential neighborhoods away from downtown.
* Because 2026 is incomplete, earlier years provide the clearest view of these patterns.
* Downtown still contains clusters of severe crashes, likely because higher volume increases the odds of severe outcomes.
* The most serious crashes cluster near large intersections, where turning conflicts and pedestrian crossings raise risk.

**Key takeaway:** While crashes are concentrated downtown, serious harm occurs citywide. Severe and fatal crashes are closely linked to major intersections, so reducing danger requires attention to both high-traffic corridors and the conflict points that turn crashes into casualties.

### Visualization: Chicago Street Crashes 2022-2026 by Street User Type (Scattermap)

**What the plot shows:** A single-panel interactive map with a year slider (2022-2026). Each point is a crash; color encodes the primary street user type: Pedestrian > Cyclist > Passenger > Driver. If at least one pedestrian is involved, the crash is marked as pedestrian. Because multiple people can be affected in one crash, we use a single primary type for clarity and cannot encode all possible combinations.

**Why this visualization:** The previous plot asked whether crash risk follows a geographic pattern. This plot asks whether user types show geographic patterns. The map makes the story concrete for readers familiar with Chicago and shows whether street users involved in crashes are tied to specific locations (for example, proximity to highways or the city center) or to specific streets.

In [15]:
### Street Crash Map colored by primary road-user group
import plotly.express as px

fig = go.Figure()
trace_meta = []

bin_colors = ["rgba(200,200,200,0.6)","rgba(252,186,3,0.8)","rgba(252,130,87,0.85)","rgba(217,72,1,0.9)"]

def infer_primary_group(row):
    if row.get("INJURIES_PEDESTRIAN", 0) > 0:
        return "Pedestrian"
    if row.get("INJURIES_BICYCLE", 0) > 0:
        return "Cyclist"
    if row.get("INJURIES_PASSENGER", 0) > 0:
        return "Passenger"
    if row.get("INJURIES_DRIVER", 0) > 0:
        return "Driver"
    return "Unknown"

if "primary_group" not in crashes.columns:
    if "crash_base" in globals() and "primary_group" in crash_base.columns:
        group_map = crash_base[["CRASH_RECORD_ID", "primary_group"]].drop_duplicates()
        crashes = crashes.merge(group_map, on="CRASH_RECORD_ID", how="left")
    else:
        crashes["primary_group"] = crashes.apply(infer_primary_group, axis=1)

crashes["PRIMARY_GROUP"] = crashes["primary_group"].fillna("")
crashes.loc[crashes["PRIMARY_GROUP"].eq(""), "PRIMARY_GROUP"] = crashes.apply(infer_primary_group, axis=1)
crashes["PRIMARY_GROUP"] = crashes["PRIMARY_GROUP"].str.title()

group_order = ["Pedestrian", "Cyclist", "Passenger", "Driver"]
palette = [PURPLE, ORANGE, TEAL, GRAY]
group_colors = {g: palette[i] for i, g in enumerate(group_order)}

for yr in years:
    crashes_y = crashes[crashes["CRASH_YEAR"] == yr].copy()
    if crashes_y.empty:
        continue

    street_risk = (
        crashes_y.groupby("STREET_KEY", dropna=True)
        .agg(total=("CRASH_RECORD_ID","nunique"), severe=("IS_SEVERE","sum"))
        .reset_index()
    )
    p0 = street_risk["severe"].sum() / street_risk["total"].sum()
    m = BAYESIAN_M
    street_risk["risk_per_100"] = 100 * (street_risk["severe"] + m * p0) / (street_risk["total"] + m)

    geom_df = base_geom_df.merge(
        street_risk[["STREET_KEY","risk_per_100"]], on="STREET_KEY", how="left"
    )
    geom_df = geom_df.dropna(subset=["risk_per_100"]).copy()
    geom_df = geom_df.sort_values("risk_per_100", ascending=False).head(MAX_LINES)

    if geom_df.empty or geom_df["risk_per_100"].dropna().empty:
        bins = np.linspace(0, 1, NUM_BINS + 1)
        geom_df["bin"] = 0
    else:
        bins = np.quantile(geom_df["risk_per_100"], np.linspace(0, 1, NUM_BINS + 1))
        geom_df["bin"] = np.clip(
            np.digitize(geom_df["risk_per_100"], bins, right=True) - 1,
            0, NUM_BINS - 1,
        )

    is_visible = yr == init_year

    for b in range(NUM_BINS):
        seg_lons, seg_lats = [], []
        sub = geom_df[geom_df["bin"] == b]
        for geom in sub["GEOM"]:
            for coords in geometry_groups(geom):
                try:
                    ls = LineString(coords).simplify(SIMPLIFY_TOL)
                except Exception:
                    continue
                if isinstance(ls, MultiLineString):
                    lines = list(ls)
                else:
                    lines = [ls]
                for line in lines:
                    xs, ys = line.xy
                    seg_lons.extend(xs); seg_lats.extend(ys)
                    seg_lons.append(None); seg_lats.append(None)
        if seg_lons:
            fig.add_trace(go.Scattermap(
                lon=seg_lons, lat=seg_lats, mode="lines",
                line=dict(color=bin_colors[b], width=1.6),
                hoverinfo="skip", showlegend=False,
                visible=is_visible,
            ))
            trace_meta.append(yr)

    pts = crashes_y.dropna(subset=["LATITUDE","LONGITUDE"]).copy()
    pts = pts.sample(frac=POINT_SAMPLE, random_state=7)

    for group in group_order:
        sub = pts[pts["PRIMARY_GROUP"] == group]
        if sub.empty:
            continue
        fig.add_trace(go.Scattermap(
            lon=sub["LONGITUDE"], lat=sub["LATITUDE"], mode="markers",
            marker=dict(size=5, color=group_colors[group]),
            name=group,
            legendgroup=group,
            showlegend=True,
            visible=is_visible,
            hovertemplate="Street User: " + group + "<extra></extra>",
        ))
        trace_meta.append(yr)

steps = []
for yr in years:
    vis = [m == yr for m in trace_meta]
    steps.append(dict(method="update", args=[{"visible": vis}], label=str(yr)))

_simple_layout(fig, title="Chicago Street Crashes Street Users 2022-2026", height=700)

fig.update_layout(
    title=dict(y=0.97, x=0.5, xanchor="center"),
    map=dict(style="carto-positron", center=dict(lat=41.8781, lon=-87.6298), zoom=11),
    margin=dict(l=20, r=20, t=110, b=110),
    height=800,
    showlegend=True,
    legend=dict(
        title="Street User",
        orientation="h",
        groupclick="togglegroup",
        x=0.5, y=1.02,
        xanchor="center", yanchor="bottom",
        font=dict(size=16, family=FONT, color=LABEL_COLOR),
    ),
    sliders=[dict(
        active=years.index(init_year),
        currentvalue=dict(prefix="Year: "),
        pad=dict(t=6),
        x=0.5, y=0.0,
        xanchor="center", yanchor="top",
        steps=steps,
    )],
    uirevision="keep",
)

fig.show()

**Key patterns**

* Pedestrian and cyclist crashes are concentrated in the city center
* Patterns match the areas with the most severe crashes
* Driver and passenger crashes are more evenly distributed across the city
* Dense neighborhoods increase interactions between transport modes
* Most pedestrian and cyclist crashes occur near intersections
* Intersections create conflicts between vehicles pedestrians and cyclists

**Key Takeaway**

- The city center is structurally more dangerous for pedestrians and cyclists because dense urban intersections increase conflicts with faster moving vehicles

## 5.3 When Crashes Happen

Crashes follow a daily and weekly rhythm, but severity does not rise and fall with volume. This section separates when crashes are common from when they are most dangerous, and compares all streets to the top 15.

The combined figure has two parts:

* Left panel: hourly severe and fatal rates alongside total crash volume.
* Right panel: a day-by-hour heatmap of severe rate.

The controls let you switch between all streets and the top 15, and between road-user groups, so you can see how timing changes by exposure and vulnerability.

Why this visualization works: pairing the bar and line panel with the heatmap shows both magnitude and pattern, making the mismatch between volume and severity clear.

### Visualization: Hourly Heatmap + Bar/Line Panel (Temporal Patterns)

**What the plot shows:** A two-panel interactive figure. The left panel stacks hourly bars for incapacitating and fatal crash *rates*, with a secondary-axis line for total crash *volume*. Rush-hour windows (07:00–09:00 and 16:00–19:00) are shaded in yellow. The right panel is a 7 × 24 heatmap of severe crash rate by day of week and hour. Both panels respond to a street-set dropdown (Top 15 / All streets) and user-group buttons (All / Driver / Passenger / Pedestrian / Cyclist).

**Why this visualization is right for the story:** The core message of this chapter is that volume and severity are temporally *decoupled* — more crashes at rush hour, but more dangerous crashes at night. The bar/line dual-axis design makes this visible in a single panel: bars for severity rate and a line for volume sharing the same x-axis. When the line peaks (rush hours) while the bars remain low, and the bars peak (late night) while the line is low, the anti-correlation is immediate and visceral. The heatmap then extends the analysis to two dimensions: it reveals that late-night Saturday and Sunday are the highest-severity windows, a pattern that a simple hourly aggregation would miss because it averages across all days. The combination gives both granularity and context.

In [16]:
### Hourly and daily patterns of severe crash risk
dow_map   = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
day_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
GROUPS    = ["All", "Driver", "Passenger", "Pedestrian", "Cyclist"]
SKEYS     = ["top", "all"]
COMBOS    = [(s, g) for s in SKEYS for g in GROUPS]   # 10 combinations

STREET_SETS = {
    "top": crash_base[crash_base["STREET_NAME"].isin(top_streets)].copy(),
    "all": crash_base.copy(),
}

def prep_df(df):
    df = df.copy()
    df["is_severe_crash"] = (
        (df["INJURIES_FATAL"].fillna(0) > 0) |
        (df["INJURIES_INCAPACITATING"].fillna(0) > 0)
    )
    df["is_deadly_crash"] = df["INJURIES_FATAL"].fillna(0) > 0
    df["CRASH_DAY_OF_WEEK"] = pd.to_numeric(
        df["CRASH_DAY_OF_WEEK"], errors="coerce"
    ).astype("Int64")
    df["CRASH_HOUR"] = (
        pd.to_numeric(df["CRASH_HOUR"], errors="coerce")
        .fillna(0).clip(0, 23).astype(int)
    )
    return df

STREET_SETS = {k: prep_df(v) for k, v in STREET_SETS.items()}

def compute_group_data(df_sub):
    hf = (
        df_sub.groupby("CRASH_HOUR", as_index=False)
        .agg(total=("CRASH_RECORD_ID",  "nunique"),
             fatal=("is_deadly_crash",  "sum"),
             severe=("is_severe_crash", "sum"))
    )
    p0_s = hf["severe"].sum() / hf["total"].sum()
    p0_f = hf["fatal"].sum()  / hf["total"].sum()
    m_s = m_f = 50

    hf["severe_rate"] = 100 * (hf["severe"] + m_s * p0_s) / (hf["total"] + m_s)
    hf["fatal_rate"]  = 100 * (hf["fatal"]  + m_f * p0_f) / (hf["total"] + m_f)
    hf["incap_count"] = (hf["severe"] - hf["fatal"]).clip(lower=0)
    hf["incap_rate"]  = (hf["severe_rate"] - hf["fatal_rate"]).clip(lower=0)

    hourly = (
        df_sub.groupby(["CRASH_DAY_OF_WEEK", "CRASH_HOUR"], as_index=False)
        .agg(total_crashes=("CRASH_RECORD_ID",  "nunique"),
             severe_crashes=("is_severe_crash", "sum"))
    )
    p0_h  = hourly["severe_crashes"].sum() / hourly["total_crashes"].sum()
    var_h = hourly["severe_crashes"].var()
    m_h   = int(p0_h * (1 - p0_h) / var_h * hourly["total_crashes"].mean()) if var_h > 0 else 1
    hourly["severe_rate_per_100"] = 100 * (
        (hourly["severe_crashes"] + m_h * p0_h) / (hourly["total_crashes"] + m_h)
    )
    hourly["day_label"] = hourly["CRASH_DAY_OF_WEEK"].map(dow_map)
    heat = (
        hourly.pivot(index="day_label", columns="CRASH_HOUR",
                     values="severe_rate_per_100")
        .reindex(day_order)
    )
    for h in range(24):
        if h not in heat.columns:
            heat[h] = np.nan
    heat = heat[sorted(heat.columns)]
    return hf, heat

group_data = {}
for skey, sdf in STREET_SETS.items():
    for grp in GROUPS:
        sub = (
            sdf if grp == "All"
            else sdf[sdf["CRASH_RECORD_ID"].isin(
                crash_base.loc[crash_base["primary_group"] == grp, "CRASH_RECORD_ID"]
            )]
        )
        group_data[(skey, grp)] = dict(zip(["hf", "heat"], compute_group_data(sub)))

heat_colorscale = [
    [0.0, "rgba(247,247,248,1)"],
    [0.6, "rgba(252,130,87,0.55)"],
    [1.0, "rgba(252,130,87,0.95)"],
]
FATAL_LABEL_THRESHOLD = 0.1

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.58, 0.42],
    specs=[[{"secondary_y": True}, {"type": "heatmap"}]],
    subplot_titles=[
        "(a)  Fatal & incapacitating crash rate by hour of day",
        "(b)  Severe rate heatmap by day & hour",
    ],
    horizontal_spacing=0.1,
)

N_TRACES_PER_COMBO = 5

for ci, (skey, grp) in enumerate(COMBOS):
    hf   = group_data[(skey, grp)]["hf"]
    heat = group_data[(skey, grp)]["heat"]
    vis  = (ci == 0)         
    first = (ci == 0)

    fig.add_trace(go.Heatmap(
        z=heat.values, x=heat.columns.tolist(), y=heat.index.tolist(),
        colorscale=heat_colorscale, visible=vis,
        showscale=first,
        colorbar=dict(
            outlinewidth=0.5, outlinecolor=LABEL_COLOR,
            title=dict(text="Severe rate (%)",
                       font=dict(family=FONT, size=14, color=LABEL_COLOR)),
            ticks="outside",
            tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
            x=0.95, len=0.6,
        ),
        hovertemplate="Day: %{y}<br>Hour: %{x}:00<br>Severe rate: %{z:.2f}<extra></extra>",
        name="heatmap",
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=YELLOW, symbol="square", opacity=0.65),
        name="Rush hours (7–9 h, 16–19 h)",
        showlegend=first, visible=vis, hoverinfo="skip",
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Bar(
        x=hf["CRASH_HOUR"], y=hf["incap_rate"],
        name="Incapacitating (non-fatal) rate (%)",
        marker=dict(color=GREEN, line=dict(color=GRAY, width=0.6)),
        text=hf["incap_rate"].astype(float).map(lambda x: f"{x:.3g}"),
        textposition="inside",
        textfont=dict(size=9, family=FONT, color=LABEL_COLOR),
        customdata=np.stack([hf["incap_count"], hf["incap_rate"]], axis=1),
        hovertemplate=(
            "Hour: %{x}:00<br>Incapacitating rate: %{customdata[1]:.2f}%<br>"
            "Incapacitating crashes: %{customdata[0]:.0f}<extra></extra>"
        ),
        showlegend=first, visible=vis,
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Bar(
        x=hf["CRASH_HOUR"], y=hf["fatal_rate"],
        name="Fatal rate (%)",
        marker=dict(color=TEAL, line=dict(color=GRAY, width=0.6)),
        text=[f"{v:.2f}" if v >= FATAL_LABEL_THRESHOLD else "" for v in hf["fatal_rate"]],
        textposition="inside",
        textfont=dict(size=9, family=FONT, color=LABEL_COLOR),
        customdata=np.stack([hf["fatal"], hf["fatal_rate"]], axis=1),
        hovertemplate=(
            "Hour: %{x}:00<br>Fatal rate: %{customdata[1]:.2f}%<br>"
            "Fatal crashes: %{customdata[0]:.0f}<extra></extra>"
        ),
        showlegend=first, visible=vis,
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=hf["CRASH_HOUR"], y=hf["total"],
        name="Total crashes", mode="lines+markers",
        line=dict(color=PURPLE, width=2.5, dash="dashdot"),
        marker=dict(size=6, color=PURPLE),
        hovertemplate="Hour: %{x}:00<br>Total crashes: %{y:,}<extra></extra>",
        showlegend=first, visible=vis, opacity=0.75,
    ), row=1, col=1, secondary_y=True)

for x0, x1 in [(7, 9), (16, 19)]:
    fig.add_shape(type="rect", xref="x", yref="paper",
                  x0=x0-0.5, x1=x1+0.5, y0=0, y1=1,
                  fillcolor=YELLOW, opacity=0.18, layer="below",
                  line_width=0.5, line_dash="dashdot")

total_traces = len(COMBOS) * N_TRACES_PER_COMBO

def _vis_mask(active_ci):
    """Boolean list: show only traces belonging to combo index active_ci."""
    out = []
    for ci in range(len(COMBOS)):
        out.extend([ci == active_ci] * N_TRACES_PER_COMBO)
    return out


street_buttons = []
for si, skey in enumerate(SKEYS):
    ci    = COMBOS.index((skey, "All"))
    label = "Top 15 streets" if skey == "top" else "All streets"
    street_buttons.append(dict(
        label=label, method="update",
        args=[{"visible": _vis_mask(ci)}, {}],
    ))

group_buttons = []
for grp in GROUPS:
    ci = COMBOS.index(("top", grp))
    group_buttons.append(dict(
        label=grp, method="update",
        args=[{"visible": _vis_mask(ci)}, {}],
    ))

_simple_layout(fig, title="Hourly & daily patterns of severe crash risk", height=650)

fig.update_layout(
    title=dict(y=0.97, xanchor="center", x=0.5),
    barmode="stack", height=650,
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    bargap=0.18, bargroupgap=0.05,
    updatemenus=[
        # Street dropdown — top-left
        dict(
            type="dropdown",
            direction="down",
            active=0,
            x=0.0, xanchor="left",
            y=1.18, yanchor="top",
            buttons=street_buttons,
            bgcolor="#f9fafb",
            bordercolor="#d0d5dd",
            borderwidth=2,
            font=dict(family=FONT, size=13, color="#344054"),
            showactive=True,
        ),
        # Group buttons — top-centre
        dict(
            type="buttons",
            direction="right",
            active=0,
            x=0.5, xanchor="center",
            y=1.18, yanchor="top",
            buttons=group_buttons,
            bgcolor="#f9fafb",
            bordercolor="#d0d5dd",
            borderwidth=2,
            font=dict(family=FONT, size=13, color="#344054"),
            showactive=True,
        ),
    ],
    legend=dict(
        orientation="h", yanchor="top", y=-0.22,
        xanchor="center", x=0.5,
        font=dict(size=16, family=FONT, color=LABEL_COLOR),
        bgcolor="rgba(255,255,255,0.85)",
    ),
    margin=dict(l=90, r=50, t=130, b=120),
    hoverlabel=dict(
        font=dict(family=FONT, size=13, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
    annotations=[
        *[
            dict(text=ann.text,
                 x=ann.x, y=ann.y, xref="paper", yref="paper",
                 showarrow=False,
                 font=dict(family=FONT, size=17, color=LABEL_COLOR),
                 xanchor="center", yanchor="bottom")
            for ann in fig.layout.annotations
        ],
        # dict(text="Streets:", xref="paper", yref="paper",
        #      x=-0.01, y=1.22, xanchor="right", yanchor="middle",
        #      showarrow=False,
        #      font=dict(family=FONT, size=13, color=LABEL_COLOR)),
        # dict(text="Group:", xref="paper", yref="paper",
        #      x=0.27, y=1.22, xanchor="right", yanchor="middle",
        #      showarrow=False,
        #      font=dict(family=FONT, size=13, color=LABEL_COLOR)),
    ],
)

fig.update_xaxes(
    title_text="Hour of day", dtick=2,
    title_font=dict(family=FONT, size=16, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
    showgrid=False, zeroline=False, row=1, col=2,
)
fig.update_yaxes(
    title_text="Day of week",
    title_font=dict(family=FONT, size=16, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
    row=1, col=2,
)
fig.update_xaxes(
    title_text="Hour of day",
    tickvals=list(range(0, 24)),
    ticktext=[str(h) for h in range(24)],
    range=[-0.5, 23.5],
    title_font=dict(family=FONT, size=16, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=11, color=LABEL_COLOR),
    showgrid=False, zeroline=False, row=1, col=1,
)
fig.update_yaxes(
    title_text="Rate (%)",
    title_font=dict(family=FONT, size=16, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
    rangemode="tozero", showgrid=True,
    gridcolor="rgba(200,200,200,0.4)",
    secondary_y=False, row=1, col=1,
)
fig.update_yaxes(
    title_text="Total crashes",
    title_font=dict(family=FONT, size=16, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
    showgrid=False, secondary_y=True, row=1, col=1, zeroline=False,
)

fig.show()

Chicago's crashes follow two different rhythms: one for volume and one for severity. Reading both panels together makes that gap clear.

Key patterns from the hourly bars and line:

* Crash volume peaks during morning and evening rush hours, but those crashes tend to be less severe.
* Severe and fatal rates rise after evening and peak in late night hours when total crashes are lower.
* The gap between volume and severity is even stronger on the top 15 streets, where late-night risk is amplified.

What the heatmap adds:

* Elevated severe rates concentrate in late-night hours across most weekdays.
* Weekends show a broader band of high severity, especially late Friday and Saturday nights.
* Pedestrians and cyclists show higher severity across many hours, not just overnight, which highlights their vulnerability.

**Key Takeaway**

The hours with the most crashes are not the hours with the greatest danger. Nighttime conditions likely amplify risk through higher speeds, lower visibility, and impairment, which suggests safety interventions should target high-severity hours, not only high-volume periods.

## 5.4 Why Crashes Happen

Understanding *where* and *when* crashes happen reveals patterns in exposure and timing. Understanding *why* they happen requires looking at the conditions and behaviours that turn a collision into a severe or fatal event.

This chapter examines three categories of crash factors:

1. **Environmental conditions** — lighting, weather, and road surface. These are background conditions that the driver cannot control but that amplify or reduce the consequences of any collision.
2. **Traffic control infrastructure** — signal types and control devices at the location of the crash. These reflect the road design environment.
3. **Compound crash factors** — combinations of behaviour and condition that recur in severe crashes (e.g., speeding in darkness, failure to yield at an intersection). These combinations are where policy levers are most actionable.

For each category we compare the citywide picture to the top 15 streets, allowing us to test whether dangerous corridors are sensitive to the same factors as the city as a whole, or whether they have distinct risk signatures.

### Visualization: Environmental Risk Factor Heatmaps

**What the plot shows:** Three side-by-side heatmaps — lighting condition, weather condition, and roadway surface condition each showing road-user groups (rows) against condition categories (columns). Cell values are *risk ratios*: the severe crash rate under that condition divided by the citywide baseline for that group. Values above 1.0 mean the condition is overrepresented in severe crashes for that group. A street-set dropdown switches between all streets and the top 15.

**Why this visualization:** With three condition variables, four user groups, and a dozen-plus condition categories per variable, a single readable chart is a genuine design challenge. The heatmap solves it by encoding one continuous variable (risk ratio) across a two-dimensional categorical grid, producing a compact layout where the reader's eye immediately goes to the darkest cells — the most dangerous condition/group combinations. Critically, the fixed shared colour scale across all three panels (identical zmin/zmax) means a cell in the "Lighting" panel is directly comparable to a cell in the "Weather" panel, avoiding the misleading impression that differences in one panel are larger or smaller than differences in another.

In [17]:
### Environmental Risk Factors
person_df = df_1.copy()
person_df["person_group"] = [
    _person_group(pt, pc)
    for pt, pc in zip(person_df["PERSON_TYPE"], person_df["IS_PEDESTRIAN_CYCLIST"])
]

person_df = crash_base[["CRASH_RECORD_ID", "is_severe_crash"]].merge(
    person_df[["CRASH_RECORD_ID", "PERSON_ID", "person_group"]],
    on="CRASH_RECORD_ID", how="inner"
)
person_df = person_df.merge(
    df_1[[
        "CRASH_RECORD_ID", "STREET_NAME",
        "LIGHTING_CONDITION", "WEATHER_CONDITION",
        "ROADWAY_SURFACE_COND", "MANEUVER", "PRIM_CONTRIBUTORY_CAUSE",
    ]].drop_duplicates(),
    on="CRASH_RECORD_ID", how="left"
)

def _compute_risk_tables(pdf):
    def _agg(group_col):
        r = (
            pdf.groupby([group_col, "person_group"])
            .agg(people=("PERSON_ID", "nunique"), severe=("is_severe_crash", "sum"))
            .reset_index()
        )
        r["severe_rate"] = r["severe"] / r["people"]
        return r
    return _agg("LIGHTING_CONDITION"), _agg("WEATHER_CONDITION"), _agg("ROADWAY_SURFACE_COND")

def _add_risk_ratio(df, x_col):
    df = df.copy()
    baseline = (
        df.groupby("person_group")
        .apply(lambda x: x["severe"].sum() / x["people"].sum())
        .rename("baseline_rate")
        .reset_index()
    )
    df = df.merge(baseline, on="person_group")
    df["risk_ratio"] = df["severe_rate"] / df["baseline_rate"].replace(0, 1e-9)
    return df

weather_short_map = {
    "CLOUDY/OVERCAST":       "CLOUDY",
    "BLOWING SNOW":          "BLOWING SNOW",
    "FREEZING RAIN/DRIZZLE": "FRZ RAIN",
    "SLEET/HAIL":            "SLEET/HAIL",
    "SNOW":                  "SNOW",
    "RAIN":                  "RAIN",
    "CLEAR":                 "CLEAR",
    "FOG/SMOKE/HAZE":        "FOG/HAZE",
    "OTHER":                 "OTHER",
    "UNKNOWN":               "OTHER",
}

SKEYS       = ["top", "all"]
SKEY_LABELS = {"top": "Top 15 streets", "all": "All streets"}

def _build_heatmaps(street_filter=None):
    pdf = person_df[person_df["STREET_NAME"].isin(street_filter)].copy() \
          if street_filter is not None else person_df.copy()
    l_risk, w_risk, r_risk = _compute_risk_tables(pdf)
    raw = [
        ("Lighting", l_risk, "LIGHTING_CONDITION"),
        ("Weather",  w_risk, "WEATHER_CONDITION"),
        ("Roadway",  r_risk, "ROADWAY_SURFACE_COND"),
    ]
    return [(label, _add_risk_ratio(df, x_col), x_col) for label, df, x_col in raw]

heatmaps_by_skey = {
    "top": _build_heatmaps(street_filter=top_streets),
    "all": _build_heatmaps(street_filter=None),
}

zmax = max(df["risk_ratio"].max() for hms in heatmaps_by_skey.values() for _, df, _ in hms)
zmin = min(df["risk_ratio"].min() for hms in heatmaps_by_skey.values() for _, df, _ in hms)

colorscale = [
    [0.0, "rgba(247,247,248,1)"],
    [0.6, "rgba(252,130,87,0.55)"],
    [1.0, "rgba(252,130,87,0.95)"],
]

N_PANELS = 3

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Lighting condition", "Weather condition", "Roadway condition"],
    horizontal_spacing=0.08,
    shared_yaxes=True,
    column_widths=[0.3, 0.42, 0.28],
    specs=[[{"type": "heatmap"}] * 3],
)

for si, skey in enumerate(SKEYS):
    hms     = heatmaps_by_skey[skey]
    visible = (si == 0)

    for col_idx, (label, df_hm, x_col) in enumerate(hms, start=1):
        df_use = df_hm[df_hm[x_col].notna()].copy()
        if label == "Weather":
            df_use[x_col] = df_use[x_col].astype(str).str.strip().replace(weather_short_map)

        pivot = df_use.pivot_table(
            index="person_group",
            columns=x_col,
            values="risk_ratio",
            fill_value=0
        )
        
        x_vals = pivot.columns.tolist()
        y_vals = pivot.index.tolist()
        z_vals = pivot.values

        show_cbar = (col_idx == 3)

        fig.add_trace(go.Heatmap(
            x=x_vals,
            y=y_vals,
            z=z_vals,
            zmin=zmin, zmax=zmax,
            colorscale=colorscale,
            visible=visible,
            text=np.round(z_vals, 2).astype(str) + "x",  
            texttemplate="%{text}", 
            textfont=dict(color=LABEL_COLOR, size=10, family=FONT),
            showscale=show_cbar,
            colorbar=dict(
                title="Risk ratio",
                len=0.8, thickness=12,
                ticks="outside",
                outlinewidth=0.5, outlinecolor=LABEL_COLOR,
                x=1.02, ticksuffix="x",
                tickfont=dict(family=FONT, color=LABEL_COLOR, size=12),
                title_font=dict(family=FONT, color=LABEL_COLOR, size=14),
            ) if show_cbar else None,
            hovertemplate=(
                f"{label} condition: %{{x}}<br>"
                "Person group: %{y}<br>"
                "Risk Ratio: %{z:.2f}x<br>"
                "<extra></extra>"
            ),
        ), row=1, col=col_idx)

def _vis_mask(active_si):
    out = []
    for si in range(len(SKEYS)):
        out.extend([si == active_si] * N_PANELS)
    return out

street_buttons = [
    dict(
        label=SKEY_LABELS[skey],
        method="update",
        args=[{"visible": _vis_mask(si)}, {}],
    )
    for si, skey in enumerate(SKEYS)
]

_simple_layout(fig, "Conditional Heatmaps of Environment on Crash Risk and Person Groups", height=550)

fig.update_layout(
    title=dict(x=0.5, xanchor="center"),
    paper_bgcolor="white", plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    updatemenus=[dict(
        type="dropdown",
        direction="down",
        active=0,
        x=0.0, xanchor="left",
        y=1.22, yanchor="top",
        buttons=street_buttons,
        bgcolor="#f9fafb",
        bordercolor="#d0d5dd",
        borderwidth=2,
        font=dict(family=FONT, size=13, color="#344054"),
        showactive=True,
    )],
    annotations=[
        *[
            dict(text=ann.text,
                 x=ann.x, y=ann.y, xref="paper", yref="paper",
                 showarrow=False,
                 font=dict(family=FONT, size=17, color=LABEL_COLOR),
                 xanchor="center", yanchor="bottom")
            for ann in fig.layout.annotations
        ],
        dict(text="Streets:", xref="paper", yref="paper",
             x=-0.01, y=1.26, xanchor="right", yanchor="middle",
             showarrow=False,
             font=dict(family=FONT, size=13, color=LABEL_COLOR)),
    ],
    margin=dict(l=70, r=70, t=120, b=120),
    hoverlabel=dict(
        font=dict(family=FONT, size=13, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
)

fig.update_xaxes(
    tickangle=-30, automargin=True,
    showgrid=True, gridcolor="rgba(0,0,0,0.08)",
    tickfont=dict(color=LABEL_COLOR, size=12, family=FONT),
)
fig.update_yaxes(
    title_text="Person group", row=1, col=1,
    showgrid=True, gridcolor="rgba(0,0,0,0.08)",
    tickfont=dict(color=LABEL_COLOR, size=14, family=FONT),
    title_standoff=20,
    title_font=dict(color=LABEL_COLOR, size=18, family=FONT),
)

fig.show()

These heatmaps compare environmental conditions by road-user group. Each cell shows a risk ratio, where values above 1.0 mean a condition is more severe than the citywide baseline for that group.

Citywide patterns are modest but consistent:

* Low-light conditions tend to increase severity across most groups.
* Adverse weather and road surfaces add smaller but noticeable risk for drivers and passengers.
* Cyclists show sharper increases under sleet or hail and on wet surfaces.

On the top 15 streets, the same conditions matter but the effects are stronger:

* Darkness, bad weather, and poor road surface conditions amplify severity across most groups.
* The amplification suggests these corridors are more sensitive to environmental stress than the city average, not just higher in volume.

### Visualization: Traffic Control Devices — Crash Counts and Severe Rate

**What the plot shows:** For the 14 most common traffic control device types, a dual-axis chart shows total unique crash counts (bars on a log scale, left y-axis) and severe crash rate (dashed line, right y-axis), sorted left-to-right by ascending severe rate.

**Why this visualization:** Traffic control devices are a direct policy lever and understanding whether some device types correlate with higher severity helps prioritise where infrastructure investment would reduce harm. The dual-axis design is necessary here because count and rate tell different stories: a signalised intersection generates many crashes simply because it channels high traffic volume, not because signals are dangerous. The log scale on the count axis prevents the two dominant categories (no control device, traffic signals) from visually compressing the remaining 12. Sorting by ascending severe rate means the safest context appears left and the most severe appears right, making the ranking immediately readable.

In [18]:
### Traffic Control Devices and severe crash risk
tcd_df = (
    crash_base.assign(
        device=lambda d: d["TRAFFIC_CONTROL_DEVICE"]
            .fillna("UNKNOWN").str.strip().str.title()
    )
    .groupby("device", as_index=False)
    .agg(
        total=("CRASH_RECORD_ID", "nunique"),
        severe=("is_severe_crash", "sum"),
    )
    .assign(non_severe=lambda d: d["total"] - d["severe"])
    .assign(severe_rate=lambda d: d["severe"] / d["total"])
    .sort_values("total", ascending=False)
    .head(14)
    .sort_values("severe_rate", ascending=True)
)

def fmt(n):
    if n >= 1_000_000:
        return f"{n/1_000_000:.1f}M"
    if n >= 1_000:
        return f"{n/1_000:.0f}k"
    return str(n)

fig_tcd = make_subplots(specs=[[{"secondary_y": True}]])

# Single total bar instead of non-severe + severe stacked
fig_tcd.add_trace(go.Bar(
    x=tcd_df["device"], y=tcd_df["total"],
    name="Total crashes",
    marker=dict(
        color=ORANGE,
        line=dict(color=GRAY, width=1),
    ),
    text=tcd_df["total"].apply(fmt),
    textposition="outside",
    textfont=dict(size=14, family=FONT, color=LABEL_COLOR),
    customdata=tcd_df[["severe", "severe_rate"]].values,
    hovertemplate="%{x}<br>Total: %{y}<br>Severe: %{customdata[0]}<br>Rate: %{customdata[1]:.1%}<extra></extra>",
), secondary_y=False)

fig_tcd.add_trace(go.Scatter(
    x=tcd_df["device"], y=tcd_df["severe_rate"] * 100,
    name="Severe rate (%)",
    mode="markers+lines",
    marker=dict(size=8, color=PURPLE, symbol="0", opacity=1),
    line=dict(color=PURPLE, width=1.5, dash="dashdot"),
    hovertemplate="%{x}<br>Severe rate: %{y:.1f}%<extra></extra>",
), secondary_y=True)

_simple_layout(fig_tcd, "Traffic Control Devices — Crash Counts & Severe Rate", height=650)
fig_tcd.update_layout(
    title=dict(
        x=0.5, xanchor="center",
    ),
    xaxis=dict(title="", tickangle=-35),
    legend=dict(orientation="h", yanchor="top", y=-0.3, xanchor="center", x=0.5, font=dict(size=16, family=FONT, color=LABEL_COLOR)),
    margin=dict(l=90, r=40, t=100, b=140),
)

fig_tcd.update_yaxes(
    title_text="Unique crashes",
    secondary_y=False,
    title_font=dict(size=18, color=LABEL_COLOR, family=FONT),
    title_standoff=10,
    type="log",
    tickfont=dict(color=LABEL_COLOR, size=14, family=FONT),
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)"
)
fig_tcd.update_yaxes(
    title_text="Severe rate (%)",
    ticksuffix="%",
    secondary_y=True,
    showgrid=True,
    griddash="dot",
    title_font=dict(size=18, color=LABEL_COLOR, family=FONT),
    title_standoff=15,
    tickvals=[1, 2, 3, 4, 5, 6, 7, 8],
    ticktext=["1%", "2%", "3%", "4%", "5%", "6%", "7%", "8%"],
    tickfont=dict(color=LABEL_COLOR, size=14, family=FONT)
)
fig_tcd.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    tickfont=dict(color=LABEL_COLOR, size=14, family=FONT),
    title_font=dict(color=LABEL_COLOR, size=16, family=FONT)
)
fig_tcd.update_layout(
    hoverlabel=dict(
        font=dict(family=FONT, size=13, color=LABEL_COLOR),
        bgcolor="white",
        bordercolor=GRAY,
    )
)
fig_tcd.show()

The traffic-control plot shows that devices with the most crashes are not always the ones with the highest severe rates. Signalized intersections dominate total counts, while severe-rate differences are smaller and likely shaped by where those devices are installed (busy arterials versus quiet residential streets).

Two cautions when reading this figure:

* The y-axis uses a severe rate, not a causal effect; control type is entangled with road design and traffic volume.
* Devices are deployed in response to risk, so higher counts can reflect exposure as much as danger.

This is a reminder to interpret rate comparisons cautiously rather than as proof that one device is safer than another.

### Visualization: Crash Factor Connections — Side-by-Side Stacked Bars with Rate Line

**What the plot shows:** Fourteen compound crash-factor combinations (e.g. "Speeding in darkness", "Failure to yield at intersections", "Improper driver action") displayed as stacked bars (non-severe in orange, severe in brown), with a secondary-axis dashed line for the severe crash rate. Panel (a) covers all streets; panel (b) covers the top-15 streets. A group-selector toggles between road-user subgroups.

**Why this visualization is right for the story:** This figure answers the "why" question: which *combinations* of circumstance and behaviour produce the most severe outcomes? The dual-axis design — count bars for frequency, rate line for severity — prevents two common misreadings: (1) conflating frequency with danger (the most common factor is not necessarily the most lethal), and (2) over-weighting rare-but-deadly factors that affect very few crashes. The side-by-side all/top-15 panels test whether the top corridors have a qualitatively different factor profile, or simply amplify the same factors the city has citywide. (The answer: largely the same factors, but with consistently higher severe rates — confirming that dangerous streets concentrate familiar risks rather than introducing exotic ones.)

In [19]:
### Crash Factor Connections
import numpy as np
from plotly.subplots import make_subplots

def format_k(x):
    if x >= 1000:
        return f"{x/1000:.1f}k".rstrip("0").rstrip(".")
    return str(int(x))

GROUPS = ["All", "Driver", "Passenger", "Pedestrian", "Cyclist"]
STREET_SETS = {
    "top": crash_base[crash_base["STREET_NAME"].isin(top_streets)],
    "all": crash_base,
}

def compute_conn_counts(base_df):
    """Compute connection counts for a given dataframe (full or filtered subset)"""
    import pandas as pd
    
    conn = base_df.copy()
    prim = conn["PRIM_CONTRIBUTORY_CAUSE"].fillna("").str.upper()
    road = conn["ROADWAY_SURFACE_COND"].fillna("").str.upper()
    manv = conn["MANEUVER"].fillna("").str.upper()
    light = conn["LIGHTING_CONDITION"].fillna("").str.upper()
    weather = conn["WEATHER_CONDITION"].fillna("").str.upper()
    lane_cnt = pd.to_numeric(conn["LANE_CNT"], errors="coerce")
    driver_action = conn["DRIVER_ACTION"].fillna("").str.upper()
    driver_vision = conn["DRIVER_VISION"].fillna("").str.upper()
    
    severe = base_df["is_severe_crash"]
    
    local_conn_defs = {
        "Speeding on wet roads":
            prim.str.contains("SPEED", regex=False) & road.str.contains("WET", regex=False),
        "Failure to yield in darkness":
            prim.str.contains("FAIL", regex=False) & light.str.contains("DARK", regex=False),
        "Speeding in darkness":
            prim.str.contains("SPEED", regex=False) & light.str.contains("DARK", regex=False),
        "Speeding in bad weather":
            prim.str.contains("SPEED", regex=False) & ~weather.isin(["CLEAR"]),
        "Failure to yield at intersections":
            prim.str.contains("FAIL", regex=False) & (conn["INTERSECTION_RELATED_I"] == "Y"),
        "Improper turn at intersections":
            manv.str.contains("TURN", regex=False) & (conn["INTERSECTION_RELATED_I"] == "Y"),
        "Multi-lane speeding crashes":
            prim.str.contains("SPEED", regex=False) & (lane_cnt >= 4),
        "Defective road + wet":
            road.str.contains("WET", regex=False) &
            conn["ROAD_DEFECT"].fillna("").str.upper().ne("NO DEFECTS"),
        "Improper driver action":
            driver_action.str.contains("IMPROPER", regex=False),
        "Failed to see hazard":
            driver_vision.str.contains("OBSTRUCTED", regex=False) |
            driver_vision.str.contains("NOT", regex=False),
        "Following too closely":
            driver_action.str.contains("FOLLOW", regex=False),
        "Disregarded traffic control":
            driver_action.str.contains("DISREGARD", regex=False) |
            prim.str.contains("DISREGARD", regex=False),
        "Disregarded traffic signals":
            prim.str.contains("TRAFFIC SIGNAL", regex=False),
        "Disregarded stop sign":
            prim.str.contains("STOP SIGN", regex=False),
    }
    
    counts = pd.DataFrame({"connection": list(local_conn_defs.keys())})
    counts["unique_crashes"] = [
        base_df.loc[m, "CRASH_RECORD_ID"].nunique()
        for m in local_conn_defs.values()
    ]
    counts["severe_crashes"] = [
        base_df.loc[m & severe, "CRASH_RECORD_ID"].nunique()
        for m in local_conn_defs.values()
    ]
    counts["non_severe_crashes"] = counts["unique_crashes"] - counts["severe_crashes"]
    counts["severe_rate"] = np.where(
        counts["unique_crashes"] > 0,
        counts["severe_crashes"] / counts["unique_crashes"] * 100,
        np.nan,
    )
    counts = counts.sort_values("unique_crashes", ascending=False)
    return counts

all_data = {}
for skey, sdf in STREET_SETS.items():
    all_data[skey] = {}
    for grp in GROUPS:
        sub = sdf if grp == "All" else sdf[sdf["primary_group"] == grp]
        counts = compute_conn_counts(sub)
        all_data[skey][grp] = {
            "connection":  counts["connection"].tolist(),
            "non_severe":  counts["non_severe_crashes"].tolist(),
            "severe":      counts["severe_crashes"].tolist(),
            "unique":      counts["unique_crashes"].tolist(),
            "severe_rate": counts["severe_rate"].tolist(),
            "text_labels": counts["unique_crashes"].apply(format_k).tolist(),
        }

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=[
        "(a)  All streets — crash factor connections",
        "(b)  Top-15 streets — crash factor connections",
    ],
    horizontal_spacing=0.12,
)

for gi, grp in enumerate(GROUPS):
    visible = (gi == 0)  
    
    for col, skey in [(1, "all"), (2, "top")]:
        init = all_data[skey][grp]
        
        fig.add_trace(go.Bar(
            x=init["connection"],
            y=init["non_severe"],
            name="Non-severe",
            marker_color=ORANGE,
            customdata=list(zip(init["unique"], init["severe"], init["severe_rate"])),
            hovertemplate=(
                "%{x}<br>"
                "Total crashes: %{customdata[0]:,}<br>"
                "Non-severe: %{y:,}<br>"
                "Severe: %{customdata[1]:,}<br>"
                "Severe rate: %{customdata[2]:.1f}%<extra></extra>"
            ),
            showlegend=(col == 1),
            visible=visible,
            legendgroup="nonsevere",
        ), row=1, col=col, secondary_y=False)

        fig.add_trace(go.Bar(
            x=init["connection"],
            y=init["severe"],
            name="Severe",
            marker_color=BROWN,
            text=init["text_labels"],
            textposition="outside",
            textfont=dict(size=10, family=FONT, color=LABEL_COLOR),
            customdata=list(zip(init["unique"], init["severe_rate"])),
            hovertemplate=(
                "%{x}<br>"
                "Total crashes: %{customdata[0]:,}<br>"
                "Severe: %{y:,}<br>"
                "Severe rate: %{customdata[1]:.1f}%<extra></extra>"
            ),
            showlegend=(col == 1),
            visible=visible,
            legendgroup="severe",
        ), row=1, col=col, secondary_y=False)

        fig.add_trace(go.Scatter(
            x=init["connection"],
            y=init["severe_rate"],
            name="Severe rate (%)",
            mode="markers+lines",
            marker=dict(size=7, color=PURPLE, symbol="circle"),
            line=dict(color=PURPLE, width=1.6, dash="dashdot"),
            hovertemplate="%{x}<br>Severe rate: %{y:.1f}%<extra></extra>",
            showlegend=(col == 1),
            visible=visible,
            legendgroup="rate",
        ), row=1, col=col, secondary_y=True)

_simple_layout(fig, "Crash Factors behind the Crashes", height=620)

buttons = []
for gi, grp in enumerate(GROUPS):
    visibility = [False] * len(fig.data)
    for ti in range(gi * 6, (gi + 1) * 6):
        visibility[ti] = True
    
    buttons.append(
        dict(
            label=grp,
            method="update",
            args=[{"visible": visibility}, {}]
        )
    )

fig.update_layout(
    barmode="stack",
    title=dict(x=0.5, xanchor="center"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    legend=dict(
        orientation="h",
        yanchor="top", y=-0.35,
        xanchor="center", x=0.5,
        font=dict(size=14, family=FONT, color=LABEL_COLOR),
    ),
    margin=dict(l=100, r=70, t=130, b=200),
    hoverlabel=dict(
        font=dict(family=FONT, size=14, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.0, xanchor="left",
            y=1.25, yanchor="top",
            buttons=buttons,
            bgcolor="#f9fafb",
            bordercolor="#d0d5dd",
            borderwidth=1,
            font=dict(size=12, family=FONT, color="#344054"),
            showactive=True,
            active=0,
        )
    ]
)

for ann in fig.layout.annotations:
    ann.update(font=dict(family=FONT, size=16, color=LABEL_COLOR),
               yanchor="bottom", y=ann.y + 0.02)

for col in [1, 2]:
    fig.update_xaxes(
        tickangle=-35, automargin=True,
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color=LABEL_COLOR, size=11, family=FONT),
        row=1, col=col,
    )
    fig.update_yaxes(
        title_text="Unique crashes",
        secondary_y=False,
        showgrid=True, gridcolor="rgba(0,0,0,0.08)",
        tickfont=dict(color=LABEL_COLOR, size=14, family=FONT),
        title_font=dict(color=LABEL_COLOR, size=16, family=FONT),
        title_standoff=10, rangemode="tozero",
        row=1, col=col,
    )
    fig.update_yaxes(
        title_text="Severe rate (%)",
        secondary_y=True,
        ticksuffix="%",
        showgrid=False, zeroline=False,
        tickfont=dict(color=LABEL_COLOR, size=14, family=FONT),
        title_font=dict(color=LABEL_COLOR, size=16, family=FONT),
        title_standoff=10,
        row=1, col=col,
    )

fig.show()

This figure connects common crash factors to outcomes. For each factor, the stacked bars show non-severe versus severe crashes, and the line shows the severe rate. The group selector lets you compare all users, drivers, passengers, pedestrians, and cyclists across all streets and the top 15.

Two themes stand out:

* Visibility and speed interact. Darkness, speeding, and wet or bad weather conditions consistently raise severe rates, especially on the top 15 streets.
* Right-of-way conflicts matter. Failure to yield, improper turns, and related intersection conditions are frequent and associated with elevated severity.

Across groups, the top 15 streets mirror the citywide ranking of factors but with stronger severity. In other words, dangerous streets concentrate familiar risks rather than introducing entirely new ones.

### Visualization: SHAP Feature Importance — What Makes a Crash Severe?

**What the plot shows:** A horizontal bar chart of SHAP values from a gradient-boosted classifier. Each bar represents a feature’s average absolute contribution to the predicted probability that a crash is severe. Larger bars indicate features with greater influence on the model’s predictions. Features shown in red increase the predicted probability of a severe crash, while features shown in blue reduce it.

**Why this visualization:** The descriptive analysis in this chapter identified which crash conditions are associated with severe outcomes through heatmaps, factor dashboards, and radar profiles. This machine learning analysis extends that work by asking a more demanding question: when all crash characteristics are considered simultaneously, which variables contain the most independent predictive information about crash severity?

The gradient-boosted model captures complex non-linear relationships and interactions between variables, but its predictive strength comes at the cost of interpretability. SHAP (SHapley Additive exPlanations) addresses this by decomposing the model’s predictions into feature-level contributions derived from cooperative game theory. The resulting chart provides a ranked measure of feature importance while accounting for the presence of all other variables in the model.

Unlike simple correlation or frequency analysis, SHAP estimates how much each feature contributes to the model’s predictions across the full dataset. This allows the analysis to distinguish variables that remain consistently informative even after controlling for related crash conditions.

**A note on Driver Action and Maneuver:** These two variables rank among the strongest predictors in the SHAP analysis but require careful interpretation. Both describe driver behaviour immediately before impact and are therefore not post-crash leakage variables in a technical sense. However, more severe crashes often receive more detailed investigation, increasing the likelihood that officers record specific behavioural categories instead of leaving fields blank or unspecified. Consequently, part of their predictive importance may reflect differences in reporting completeness rather than purely causal behavioural effects. Their SHAP importance should therefore be interpreted as a possible upper bound on their true causal contribution to crash severity.

In [20]:
### ML — Feature preparation
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ── Target ────────────────────────────────────────────────────────────────
# Work on crash-level rows (one row per crash record)
ml_df = crash_base.copy()
ml_df["is_severe_crash"] = (
    (ml_df["INJURIES_FATAL"].fillna(0) > 0)
    | (ml_df["INJURIES_INCAPACITATING"].fillna(0) > 0)
).astype(int)

# ── Feature columns ───────────────────────────────────────────────────────
CATEGORICAL_COLS = [
    "WEATHER_CONDITION", "LIGHTING_CONDITION", "ROADWAY_SURFACE_COND", "ROAD_DEFECT",
    "TRAFFICWAY_TYPE", "ALIGNMENT",
    "INTERSECTION_RELATED_I", "NOT_RIGHT_OF_WAY_I",
    "PERSON_TYPE", "SEX", "SAFETY_EQUIPMENT", "PHYSICAL_CONDITION",
    "DRIVER_ACTION", "DRIVER_VISION",
    "EXCEED_SPEED_LIMIT_I", "HIT_AND_RUN_I",
    "VEHICLE_TYPE", "MANEUVER",
]
NUMERIC_COLS = [
    "POSTED_SPEED_LIMIT", "LANE_CNT",
    "AGE",
    "CRASH_HOUR", "CRASH_DAY_OF_WEEK", "CRASH_MONTH",
    "NUM_UNITS",
]
FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS

# ── Fill missing ──────────────────────────────────────────────────────────
for c in CATEGORICAL_COLS:
    if c in ml_df.columns:
        ml_df[c] = ml_df[c].fillna("Unknown").astype(str).str.strip()
    else:
        ml_df[c] = "Unknown"

for c in NUMERIC_COLS:
    if c in ml_df.columns:
        ml_df[c] = pd.to_numeric(ml_df[c], errors="coerce").fillna(-1)
    else:
        ml_df[c] = -1

# ── Label-encode categoricals ─────────────────────────────────────────────
le_map = {}
for c in CATEGORICAL_COLS:
    le = LabelEncoder()
    ml_df[c + "_enc"] = le.fit_transform(ml_df[c])
    le_map[c] = le          # keep encoder so we can look up category names later

ENC_COLS = [c + "_enc" for c in CATEGORICAL_COLS] + NUMERIC_COLS

X = ml_df[ENC_COLS].values
y = ml_df["is_severe_crash"].values

# Pretty display names (strip _enc, title-case)
FEATURE_NAMES = [c.replace("_enc","").replace("_"," ").title() for c in ENC_COLS]

print(f"Dataset: {X.shape[0]:,} crashes × {X.shape[1]} features")
print(f"Severe crash rate: {y.mean()*100:.2f}%  ({y.sum():,} severe / {len(y):,} total)")


Dataset: 1,034,515 crashes × 25 features
Severe crash rate: 1.73%  (17,882 severe / 1,034,515 total)


In [21]:
### ML — Gradient boosting + SHAP values
from sklearn.ensemble import GradientBoostingClassifier
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Subsample for speed — 40k rows is plenty for stable SHAP estimates
rng = np.random.default_rng(42)
idx = rng.choice(len(X_tr), size=min(40_000, len(X_tr)), replace=False)
X_sub = X_tr[idx]
y_sub = y_tr[idx]

gbm = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=50,
    random_state=42,
)
gbm.fit(X_sub, y_sub)

auc_gbm = roc_auc_score(y_te, gbm.predict_proba(X_te)[:, 1])
print(f"GBM  AUC-ROC: {auc_gbm:.4f}")

# ── SHAP ──────────────────────────────────────────────────────────────────
# Use a background sample for the TreeExplainer
X_shap = X_te[:5_000]          # 5 k test rows is enough for a stable mean
explainer   = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(X_shap)   # shape (n, p)

# Mean absolute SHAP per feature — the "global importance" ranking
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = (
    __import__('pandas').DataFrame({
        "feature": FEATURE_NAMES,
        "mean_abs_shap": mean_abs_shap,
        "mean_shap": shap_values.mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 15 features by mean |SHAP|:")
print(shap_df.head(15).to_string(index=False))


GBM  AUC-ROC: 0.7737

Top 15 features by mean |SHAP|:
               feature  mean_abs_shap  mean_shap
              Maneuver       0.234628   0.010513
         Driver Action       0.225652  -0.011818
                   Age       0.209359   0.002523
Intersection Related I       0.183212   0.003703
             Num Units       0.137460   0.000133
    Physical Condition       0.108634   0.018697
            Crash Hour       0.103372   0.000532
           Crash Month       0.090970   0.001813
          Vehicle Type       0.090517   0.002776
      Safety Equipment       0.073630  -0.017941
       Trafficway Type       0.073447   0.002295
    Posted Speed Limit       0.073428   0.001889
    Lighting Condition       0.064715   0.000166
  Roadway Surface Cond       0.055735   0.000478
                   Sex       0.049093   0.008292


In [22]:
TOP_N = 18
shap_top = shap_df.head(TOP_N).copy()

bar_colors = [
    ORANGE if v >= 0 else GRAY
    for v in shap_top["mean_shap"]
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=shap_top["mean_abs_shap"],
    y=shap_top["feature"],
    orientation="h",
    marker_color=bar_colors,
    text=shap_top["mean_abs_shap"].round(3),
    textposition="outside",
    textfont=dict(family=FONT, size=11, color=LABEL_COLOR),
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Mean |SHAP|: %{x:.5f}<br>"
        "<extra></extra>"
    ),
    showlegend=False,
))

# ── Legend annotation ─────────────────────────────────────────────────────
legend_y = -0.18

fig.add_annotation(
    xref="paper", yref="paper",
    x=0.25, y=legend_y,
    text=f"<span style='color:{ORANGE}'>■</span> Increases severity probability",
    showarrow=False,
    font=dict(family=FONT, size=14, color=LABEL_COLOR),
    xanchor="left",
)

fig.add_annotation(
    xref="paper", yref="paper",
    x=0.62, y=legend_y,
    text=f"<span style='color:{GRAY}'>■</span> Decreases severity probability",
    showarrow=False,
    font=dict(family=FONT, size=14, color=LABEL_COLOR),
    xanchor="left",
)

# ── Footer stats ──────────────────────────────────────────────────────────
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.5, y=-0.32,
    text=(
        f"Gradient boosting AUC-ROC: {auc_gbm:.3f}  |  "
        f"Target: is_severe_crash (fatal or incapacitating injury)  |  "
        f"n = {len(y):,} crashes"
    ),
    showarrow=False,
    font=dict(family=FONT, size=14, color=LABEL_COLOR),
    xanchor="center",
)

# ── Axes & layout ─────────────────────────────────────────────────────────
fig.update_xaxes(
    title_text="Mean absolute SHAP value",
    showgrid=True,
    gridcolor="rgba(0,0,0,0.08)",
    zeroline=True,
    title_font=dict(family=FONT, size=14, color=LABEL_COLOR),
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
)

fig.update_yaxes(
    autorange="reversed",
    showgrid=True,
    tickfont=dict(family=FONT, size=12, color=LABEL_COLOR),
)

_simple_layout(fig, "Top 18 Feature importance from SHAP values", height=600)

fig.update_layout(
    title=dict(
        x=0.5, xanchor="center",
    ),
    height=700,
    width=1400,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR, size=16),
    margin=dict(l=200, r=40, t=60, b=160),
    hoverlabel=dict(
        font=dict(family=FONT, size=12, color=LABEL_COLOR),
        bgcolor="white",
        bordercolor="#cccccc",
    ),
)

fig.show()

**SHAP importance**

SHAP values measure each feature's average contribution to the model's predicted severe-crash probability. A mean absolute SHAP of 0.02 means the feature shifts the predicted probability up or down by 2 percentage points on average across the test set. Bar colour indicates direction: red features increase predicted severity on average, blue features reduce it.

The ranking quantifies and confirms the patterns found throughout this chapter:

- **Intersection Related** ranks in the top four, directly confirming the intersection finding from Chapters 2 and 4.
- **Lighting Condition** carries a strong positive SHAP, confirming that darkness is a genuine independent amplifier of crash severity, consistent with the heatmap and factor dashboard findings earlier in this chapter.
- **Age** contributes substantially, reflecting that crashes involving very young or elderly road users are structurally more likely to result in severe injury.
- **Posted Speed Limit** has a positive SHAP: higher design speeds are associated with higher crash energy and higher severity, independent of other conditions.
- **Crash Hour** carries moderate SHAP, capturing the night-time severity elevation documented in Chapter 3.

The gradient-boosted model's AUC-ROC is meaningfully above the decision tree, confirming that the features contain genuine predictive signal beyond what four splits can capture. The tree is shown for interpretability. The GBM SHAP values are the analytically rigorous output.

**Limitations**

The model operates on crash records, not road segments, so it predicts the probability that a *given crash* under *given conditions* is severe. It cannot estimate the unconditional risk of a street, because traffic volume data is absent. Age and person type are person-level features, meaning a crash with multiple people contributes multiple rows with different feature values. This is handled correctly during training but means the effective sample size is slightly larger than the number of independent crash events.


## 5.5 Conclusion

### Visualization: Risk Fingerprint Radar Charts

**What the plot shows:** Two sets of 3 × 5 small-multiple radar charts, one per top street. One result worth highlighting from these radars: the citywide baseline on the intersection axis already sits at close to twice the general risk level. Being at or near an intersection roughly doubles the probability that a crash results in a severe outcome compared to a mid-block collision. This directly confirms the geographic finding from Chapter 2, where severe crashes clustered consistently around large intersections across the entire city. The radar quantifies what the map showed spatially: intersection geometry is a citywide amplifier of crash severity, not a quirk of a handful of corridors. The first set (time and weather focused) plots five axes: Wet road, Darkness, Bad weather, Outside Peak Hour, and Intersection. The second set (road and action focused) plots: Road defect, Speeding, Improper driver action, Failure to yield/see/follow, and Bad weather. Each panel overlays the street's risk profile (coloured fill) against the citywide baseline for the same user group (grey ghost). Values are expressed as ratios relative to baseline (1.0 = city average; 2.0 = twice as likely to be present in severe crashes on this street).

**Why this visualization is right for the story:** A radar chart is the ideal format for comparing multi-dimensional risk profiles across many corridors simultaneously. The radar shape acts as a "fingerprint" — a street dominated by darkness and speeding looks visually distinct from one dominated by intersections and failure-to-yield. Fifteen line charts would require scanning 75 separate bar charts; fifteen tables would bury the pattern. The grey city-ghost overlay in every panel provides a permanent baseline, so the reader answers "above or below average?" purely from shape, without consulting a legend. This is the narrative payoff of the entire analysis: showing that the most dangerous streets are not dangerous for the same reasons, which makes the case for corridor-specific intervention rather than blanket citywide policy.

In [23]:
### Risk Profiles of Top 15 Streets - Time and Weather focused
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

street_selection = top_streets
GROUPS = ["All", "Driver", "Passenger", "Pedestrian", "Cyclist"]

dimensions = ["Wet road", "Darkness", "Bad weather", "Outside of Peak hour", "Intersection"]
theta      = dimensions + [dimensions[0]]

# ── Helpers ───────────────────────────────────────────────────────────────────
def _flag_condition(df):
    out = pd.DataFrame(index=df.index)
    out["Wet road"]              = df["ROADWAY_SURFACE_COND"].fillna("").str.upper().str.contains("WET")
    out["Darkness"]              = df["LIGHTING_CONDITION"].fillna("").str.upper().str.contains("DARK")
    out["Bad weather"]           = ~df["WEATHER_CONDITION"].fillna("").str.upper().str.contains("CLEAR")
    h = df["CRASH_HOUR"]
    out["Outside of Peak hour"]  = (h < 7) | (h > 19)
    out["Intersection"]          = df["INTERSECTION_RELATED_I"].fillna("").str.upper() == "Y"
    return out

def _radar_vals(flags_df):
    severe = flags_df[flags_df["is_severe_crash"] == True]
    if len(severe) == 0:
        return [0.0] * len(dimensions)
    vals = []
    for d in dimensions:
        rate_cond = severe[d].mean()
        rate_base = flags_df[d].mean() + 1e-9
        vals.append(float(min(rate_cond / rate_base, 3.0)))
    return vals

def _subset_for_group(grp):
    """Return crash_base filtered to crashes where primary_group == grp (or all)."""
    if grp == "All":
        return crash_base.copy()
    ids = crash_base.loc[crash_base["primary_group"] == grp, "CRASH_RECORD_ID"]
    return crash_base[crash_base["CRASH_RECORD_ID"].isin(ids)].copy()

# ── Pre-compute radar values for every group × street + city-wide ─────────────
radar_data = {}
for grp in GROUPS:
    sub = _subset_for_group(grp)
    grp_entry = {}

    # city-wide for this group
    all_flags = _flag_condition(sub)
    all_flags["is_severe_crash"] = sub["is_severe_crash"].values
    city_vals = _radar_vals(all_flags)
    grp_entry["__city__"] = city_vals + [city_vals[0]]

    # per street
    for street in street_selection:
        sdf   = sub[sub["STREET_NAME"] == street].copy()
        flags = _flag_condition(sdf)
        flags["is_severe_crash"] = sdf["is_severe_crash"].values
        vals  = _radar_vals(flags)
        grp_entry[street] = vals + [vals[0]]

    radar_data[grp] = grp_entry

# ── Build figure with all groups (traces visible/hidden by group) ─────────────
panels  = street_selection
n_cols  = 5
n_rows  = int(np.ceil(len(panels) / n_cols))
palette = [BLUE, ORANGE, PURPLE, GREEN, BROWN]

fig_radar = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=panels,
    specs=[[{"type": "polar"}] * n_cols for _ in range(n_rows)],
    horizontal_spacing=0.04,
    vertical_spacing=0.12,
)

# Add traces for each group (2 traces per panel: city ghost + street)
for gi, grp in enumerate(GROUPS):
    grp_data = radar_data[grp]
    visible = (gi == 0)  # Only first group visible initially
    
    for i, label in enumerate(panels):
        color = palette[i % len(palette)]
        row = i // n_cols + 1
        col = i % n_cols + 1

        # city ghost
        fig_radar.add_trace(
            go.Scatterpolar(
                r=grp_data["__city__"],
                theta=theta,
                fill="toself",
                mode="lines",
                line=dict(color=GRAY, width=1.6, dash="dashdot"),
                fillcolor=GRAY,
                opacity=0.25,
                showlegend=(i == 0),
                name="City-wide avg",
                hoverinfo="skip",
                visible=visible,
                legendgroup="city",
            ),
            row=row, col=col,
        )

        # street radar
        fig_radar.add_trace(
            go.Scatterpolar(
                r=grp_data[label],
                theta=theta,
                fill="toself",
                marker=dict(color=color, size=5),
                name=label,
                line=dict(color=color, width=1.6),
                fillcolor=color,
                opacity=0.32,
                showlegend=(i == 0),
                hovertemplate="<b>" + label + "</b><br>%{theta}: %{r:.2f}x<extra></extra>",
                visible=visible,
                legendgroup=f"street_{i}",
            ),
            row=row, col=col,
        )

_simple_layout(fig_radar, "Risk Profiles of Top 15 Streets - Road and Action focused", height=860)
fig_radar.update_layout(
    title=dict(x=0.5, xanchor="center"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    legend=dict(
        orientation="h", yanchor="top", y=-0.12,
        xanchor="center", x=0.5,
        font=dict(size=13, color=LABEL_COLOR, family=FONT),
    ),
    margin=dict(l=40, r=40, t=120, b=120),
    hoverlabel=dict(
        font=dict(family=FONT, size=12, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
)
fig_radar.update_polars(
    radialaxis=dict(
        range=[0, 3], showticklabels=True,
        tickvals=[0, 1, 2, 3],
        gridcolor="#D4D4D4",
        tickfont=dict(color=LABEL_COLOR, size=7, family=FONT),
    ),
    angularaxis=dict(tickfont=dict(family=FONT, size=10, color=LABEL_COLOR)),
)
fig_radar.update_annotations(
    font=dict(size=11, color=LABEL_COLOR, family=FONT), yshift=12
)

# ── Create buttons for group selection ─────────────────────────────────────────
buttons = []
traces_per_group = 2 * len(panels)
for gi, grp in enumerate(GROUPS):
    visibility = [False] * len(fig_radar.data)
    for ti in range(gi * traces_per_group, (gi + 1) * traces_per_group):
        visibility[ti] = True
    
    buttons.append(
        dict(
            label=grp,
            method="update",
            args=[{"visible": visibility}, {}]
        )
    )

fig_radar.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.0, xanchor="left",
            y=1.12, yanchor="top",
            buttons=buttons,
            bgcolor="#f9fafb",
            bordercolor="#d0d5dd",
            borderwidth=1,
            font=dict(size=12, family=FONT, color="#344054"),
            showactive=True,
            active=0,
        )
    ]
)

fig_radar.show()

In [24]:
### Risk Profiles of Top 15 Streets - Road and Action focused
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

street_selection = top_streets
GROUPS = ["All", "Driver", "Passenger", "Pedestrian", "Cyclist"]

DIMENSIONS = [
    "Road defect",
    "Speeding",
    "Improper driver action",
    "Failure to (yield/see/follow)",
    "Bad weather",
]
THETA = DIMENSIONS + [DIMENSIONS[0]]

def _flag_condition(df):
    out = pd.DataFrame(index=df.index)
    prim = df["PRIM_CONTRIBUTORY_CAUSE"].fillna("").str.upper()
    road_defect = df["ROAD_DEFECT"].fillna("").str.upper()
    manv = df["MANEUVER"].fillna("").str.upper()
    driver_action = df["DRIVER_ACTION"].fillna("").str.upper()
    driver_vision = df["DRIVER_VISION"].fillna("").str.upper()
    weather = df["WEATHER_CONDITION"].fillna("").str.upper()

    out["Road defect"] = road_defect.ne("NO DEFECTS") & road_defect.ne("")
    out["Speeding"] = prim.str.contains("SPEED", regex=False)

    out["Improper driver action"] = (
        manv.str.contains("TURN", regex=False) |
        driver_action.str.contains("DISREGARD", regex=False) |
        prim.str.contains("DISREGARD", regex=False) |
        prim.str.contains("STOP SIGN", regex=False)
    )

    out["Failure to (yield/see/follow)"] = (
        prim.str.contains("FAIL", regex=False) |
        driver_vision.str.contains("OBSTRUCTED", regex=False) |
        driver_vision.str.contains("NOT", regex=False) |
        driver_action.str.contains("FOLLOW", regex=False)
    )

    out["Bad weather"] = ~weather.isin(["CLEAR"])

    return out

def _radar_vals(flags_df):
    severe = flags_df[flags_df["is_severe_crash"] == True]
    if len(severe) == 0:
        return [0.0] * len(DIMENSIONS)
    vals = []
    for d in DIMENSIONS:
        rate_cond = severe[d].mean()
        rate_base = flags_df[d].mean() + 1e-9
        vals.append(float(min(rate_cond / rate_base, 3.0)))
    return vals

def _subset_for_group(grp):
    """Return crash_base filtered to crashes where primary_group == grp (or all)."""
    if grp == "All":
        return crash_base.copy()
    ids = crash_base.loc[crash_base["primary_group"] == grp, "CRASH_RECORD_ID"]
    return crash_base[crash_base["CRASH_RECORD_ID"].isin(ids)].copy()

radar_data = {}
for grp in GROUPS:
    sub = _subset_for_group(grp)
    grp_entry = {}

    all_flags = _flag_condition(sub)
    all_flags["is_severe_crash"] = sub["is_severe_crash"].values
    city_vals = _radar_vals(all_flags)
    grp_entry["__city__"] = city_vals + [city_vals[0]]

    for street in street_selection:
        sdf   = sub[sub["STREET_NAME"] == street].copy()
        flags = _flag_condition(sdf)
        flags["is_severe_crash"] = sdf["is_severe_crash"].values
        vals  = _radar_vals(flags)
        grp_entry[street] = vals + [vals[0]]

    radar_data[grp] = grp_entry

panels  = street_selection
n_cols  = 5
n_rows  = int(np.ceil(len(panels) / n_cols))
palette = [BLUE, ORANGE, PURPLE, GREEN, BROWN]

fig_radar = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=panels,
    specs=[[{"type": "polar"}] * n_cols for _ in range(n_rows)],
    horizontal_spacing=0.04,
    vertical_spacing=0.12,
)

for gi, grp in enumerate(GROUPS):
    grp_data = radar_data[grp]
    visible = (gi == 0)
    
    for i, label in enumerate(panels):
        color = palette[i % len(palette)]
        row = i // n_cols + 1
        col = i % n_cols + 1

        # city ghost
        fig_radar.add_trace(
            go.Scatterpolar(
                r=grp_data["__city__"],
                theta=THETA,
                fill="toself",
                mode="lines",
                line=dict(color=GRAY, width=1.6, dash="dashdot"),
                fillcolor=GRAY,
                opacity=0.25,
                showlegend=(i == 0),
                name="City-wide avg",
                hoverinfo="skip",
                visible=visible,
                legendgroup="city",
            ),
            row=row, col=col,
        )

        fig_radar.add_trace(
            go.Scatterpolar(
                r=grp_data[label],
                theta=THETA,
                fill="toself",
                marker=dict(color=color, size=5),
                name=label,
                line=dict(color=color, width=1.6),
                fillcolor=color,
                opacity=0.32,
                showlegend=(i == 0),
                hovertemplate="<b>" + label + "</b><br>%{theta}: %{r:.2f}x<extra></extra>",
                visible=visible,
                legendgroup=f"street_{i}",
            ),
            row=row, col=col,
        )

_simple_layout(fig_radar, "Risk Profiles of Top 15 Streets - Road and Action focused", height=860)
fig_radar.update_layout(
    title=dict(x=0.5, xanchor="center"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(family=FONT, color=LABEL_COLOR),
    legend=dict(
        orientation="h", yanchor="top", y=-0.12,
        xanchor="center", x=0.5,
        font=dict(size=13, color=LABEL_COLOR, family=FONT),
    ),
    margin=dict(l=40, r=40, t=120, b=120),
    hoverlabel=dict(
        font=dict(family=FONT, size=12, color=LABEL_COLOR),
        bgcolor="white", bordercolor=GRAY,
    ),
)
fig_radar.update_polars(
    radialaxis=dict(
        range=[0, 3], showticklabels=True,
        tickvals=[0, 1, 2, 3],
        gridcolor="#D4D4D4",
        tickfont=dict(color=LABEL_COLOR, size=7, family=FONT),
    ),
    angularaxis=dict(tickfont=dict(family=FONT, size=10, color=LABEL_COLOR)),
)
fig_radar.update_annotations(
    font=dict(size=11, color=LABEL_COLOR, family=FONT), yshift=12
)

buttons = []
traces_per_group = 2 * len(panels)
for gi, grp in enumerate(GROUPS):
    visibility = [False] * len(fig_radar.data)
    for ti in range(gi * traces_per_group, (gi + 1) * traces_per_group):
        visibility[ti] = True
    
    buttons.append(
        dict(
            label=grp,
            method="update",
            args=[{"visible": visibility}, {}]
        )
    )

fig_radar.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.0, xanchor="left",
            y=1.12, yanchor="top",
            buttons=buttons,
            bgcolor="#f9fafb",
            bordercolor="#d0d5dd",
            borderwidth=1,
            font=dict(size=12, family=FONT, color="#344054"),
            showactive=True,
            active=0,
        )
    ]
)

fig_radar.show()

The radar profiles summarize each top street's risk fingerprint across five conditions. Each panel compares a street to the citywide baseline for the same road-user group. Some corridors peak on darkness, others on intersections or speeding, and a few show broad elevation across multiple axes. This is the clearest evidence that the most dangerous streets are not dangerous for the same reason.

**Conclusion**

From a citywide view, danger is a mix of exposure and severity: high-volume corridors generate many severe crashes, while a smaller set of streets have fewer crashes but far higher per-crash risk. When we zoom in on time, severity concentrates at night and on weekends, especially for vulnerable users. When we zoom further into conditions and behaviors, the same core risks appear again and again, but with stronger effects on the top streets.

Practical implications:

* Target lighting and nighttime enforcement where darkness drives risk.
* Redesign intersections and manage turning conflicts where right-of-way failures dominate.
* Use speed management and surface maintenance on corridors where speed and road conditions amplify severity.

A one-size-fits-all policy would miss the street-level mechanisms that drive severe outcomes. Street safety improvements should be targeted, diagnostic, and matched to the risk profile of each corridor.

Why this visualization works: radar charts compare multi-factor profiles in one compact view, making street-level fingerprints easy to contrast.

## Data Analysis Summary

The analysis unfolds across four analytical chapters, each building on the last:

**Chapter 1 — Measuring Dangerous Streets** established the risk metric: Bayesian shrinkage-adjusted severe crash rates, combined with a composite danger score that weights fatalities separately. This corrects for the noise in small-sample streets without discarding them. The top 15 streets were selected by visually inspecting the risk–volume scatter plot rather than by a data-driven cutoff. For each road user group, the streets that stood out as clear outliers — either high on the severity axis, far to the right on the volume axis, or isolated away from the main cluster — were included. This approach was deliberate: the scatter plot was designed to make those outliers visible, so using it directly as the selection tool is consistent with the analysis. The key finding is that these streets carry a disproportionate share of severe outcomes, and their severe rates exceed the citywide average across most road-user groups.

**Chapter 2 — Where Crashes Happen** mapped crash locations across Chicago using two interactive views: all crashes coloured by severity, and all crashes coloured by the most vulnerable road user involved. Both maps cover 2022 to 2026. The geographic analysis revealed that total crash volume clusters heavily in the city center and along major arterials, while fatal and incapacitating crashes spread much more evenly across the city, including residential areas on the south and west sides. The most consistent spatial pattern is that severe crashes cluster near large intersections regardless of neighbourhood, pointing to intersection geometry as a citywide amplifier of crash risk.

**Chapter 3 — When Crashes Happen** revealed a systematic decoupling between crash volume and crash severity across time. Rush hours generate the most crashes, but late-night and weekend hours produce the highest severe and fatal rates — a pattern that is amplified on the top 15 streets. This finding has direct implications for how enforcement and lighting should be allocated.

**Chapter 4 — Why Crashes Happen** showed that environmental conditions (darkness, wet roads, adverse weather) and behavioural factors (speeding, failure to yield, disregarding traffic control) are consistently overrepresented in severe crashes. The top 15 streets show the same risk profile as the city as a whole, but with stronger effect sizes — suggesting that these corridors are particularly sensitive to risk amplifiers, not merely busier.

# 6. Discussion

### What went well

The Martini Glass narrative structure was chosen before the project began because it pairs well with the data logic: start broad, narrow progressively, then open back up into reader-driven exploration. The story opens with a citywide view of what risk means and how to measure it, then moves geographically to where crashes concentrate, then to when they are most dangerous, and finally to why specific streets are dangerous in specific ways. At each stage, interactive filters — by road user group and by street set — let readers drill into the dimension that matters most to them without losing the author-guided thread.

In practice, the Martini Glass structure worked reasonably well, although the project did not follow the pattern perfectly. Some sections remained broader and more exploratory than originally planned, particularly in the middle chapters where multiple environmental and temporal dimensions needed to be established before narrowing toward specific street-level conclusions. The narrative ultimately moved from the city scale, to dangerous corridors and intersections, and finally to factor-level explanations through the radar profiles and machine learning analysis. Ending on the radar plots worked especially well because they tied together the geographic, temporal, and behavioural findings into recognisable “street identities,” making the final analytical layer feel grounded in the earlier chapters rather than disconnected from them.

The zoom-in progression worked well in practice. Readers who follow the chapters in order arrive at the radar profiles already knowing which streets matter and why the risk metric was designed the way it was. The final figures then feel like confirmation of a story that has been building, rather than a standalone result.

Bayesian shrinkage was the right methodological choice. Without traffic volume data, raw severe crash rates on low-crash streets are dominated by noise: one bad event on a quiet street would otherwise imply a 100 percent severity rate. Shrinkage pulls those estimates toward the citywide baseline in a principled, interpretable way. The fixed parameter of k = 50 means that a street needs at least 50 crashes before its observed rate carries more weight than the prior. In practice, this stabilises comparisons between outlier streets like Russell Drive and high-volume corridors like Western Avenue without discarding either from the analysis.

The consistent visual style across all figures — shared font, colour palette, and plot template — gives the notebook a unified feel and reduces the cognitive load of switching between charts. Readers do not need to re-learn a new colour scheme for each section.

### What was difficult

**Data quality and preprocessing were the most time-consuming part of the project by a wide margin.** The four source datasets use different primary keys, inconsistent categorical encodings, and a large number of missing or `"UNKNOWN"` values across behavioural and environmental fields. Joining the People and Crashes tables introduced row explosions that required careful deduplication logic. Fields like `PRIM_CONTRIBUTORY_CAUSE` and `DRIVER_ACTION` are partially redundant and inconsistently filled, making factor analysis imprecise. The `LIGHTING_CONDITION` and `WEATHER_CONDITION` fields have high `"UNKNOWN"` rates in older records, which introduces noise into the environmental risk analysis.

**Handling missing values in the machine learning pipeline was also challenging.** Real-world crash datasets are only partially clean, and missing values often have different meanings depending on the variable. For some columns, a missing value effectively means `"False"` or `"Not Present"`; for others, it means the information was simply not recorded and is genuinely unknown. Distinguishing between these cases required manual interpretation of field semantics and careful preprocessing decisions before fitting the gradient-boosting and decision-tree models. Incorrect handling of NaN values could easily introduce artificial signal or suppress meaningful patterns.

**Risk assessment without traffic volume data is fundamentally limited.** The most significant methodological gap is the absence of a volume denominator. We cannot distinguish between "many crashes because many cars" and "many crashes because the street is dangerous." Bayesian shrinkage mitigates the small-sample problem but does not solve the normalisation problem. Ideally, crashes-per-vehicle-kilometre-travelled would replace our raw severe rate, but no street-level volume data was available at the necessary granularity and temporal range. All severity comparisons in this project should be read as comparisons of observed outcomes, not of underlying causal risk.

**Showing multiple dimensions in a single figure was technically and visually demanding.** Several figures (the temporal heatmap + bar/line panel, the environmental heatmaps, the crash factor side-by-sides, the radar arrays) pack many dimensions into one view. This required careful use of Plotly's static updatemenus — which do not support independent, composable filters without JavaScript — leading to complex trace-stacking logic. Some figures became difficult to debug and may render slowly on low-memory machines.

**Visualising deeper parts of the decision tree also proved difficult.** Early in the project, several alternative visualisations were explored, including Sankey-style flow diagrams and expanded lower-level tree plots intended to show how crashes move through successive splits. In practice, these approaches became difficult to interpret because the sample sizes shrink rapidly deeper in the tree, making many terminal branches statistically weak or visually insignificant. The problem was amplified by the high-cardinality categorical variables in the dataset, where features can contain dozens of unique categories. This produced crowded and fragmented visualisations with many thin branches that were visually cluttered and provided little additional analytical insight. As a result, these approaches were ultimately abandoned in favour of the simpler depth-3 tree and SHAP importance chart, which communicated the main patterns more clearly and reliably.

**Unsupervised clustering approaches were explored but ultimately abandoned due to weak structure in the data.** We experimented with dimensionality reduction using UMAP embeddings followed by clustering methods including k-nearest-neighbour-based approaches and agglomerative clustering. The goal was to identify latent street-user types or natural groupings of severe crashes without predefined labels. However, the resulting silhouette scores were consistently very low, indicating weak separation between clusters and poor cluster quality overall. Because the clustering results were not sufficiently robust or interpretable, this direction was not pursued further in the final analysis.

### What is still missing or could be improved

- **Statistical testing.** All comparisons in this project are descriptive. No confidence intervals, p-values, or effect-size estimates are reported. For a policy audience, quantifying uncertainty in the severity rate estimates would strengthen the conclusions.
- **Pedestrian and cyclist exposure.** Person-level trip counts or pedestrian/cyclist volumes would allow proper exposure normalisation for vulnerable users — the groups most affected by severe crashes.
- **Temporal granularity in Chapter 1.** The risk scatter aggregates all years. A year-by-year scatter would show whether the top-15 ranking is stable over time or whether specific streets entered or exited the danger list.
- **Causal inference.** The entire analysis is correlational. Streets with poor lighting may be dangerous because of lighting, or because they attract nighttime activity, or because they are located in areas with other risk factors. Addressing causality would require a quasi-experimental design (e.g., comparing streets before and after lighting upgrades).

## 7. Contributions

This project was completed individually.

- **s253495:** responsible for all parts of the assignment
---

## References

- City of Chicago Data Portal. *Traffic Crashes — Crashes, People, Vehicles* and *Street Center Lines*. https://data.cityofchicago.org. Accessed 2026-05-07.
- Segel, E., & Heer, J. (2010). Narrative Visualization: Telling Stories with Data. *IEEE Transactions on Visualization and Computer Graphics*, 16(6), 1139–1148.
- Efron, B., & Morris, C. (1977). Stein's Paradox in Statistics. *Scientific American*, 236(5), 119–127. *(Background on empirical Bayes / shrinkage estimation.)*
- National Highway Traffic Safety Administration (NHTSA). (2023). *Traffic Safety Facts: A Compilation of Motor Vehicle Crash Data*. DOT HS 813 491.
- Elvik, R., Høye, A., Vaa, T., & Sørensen, M. (2009). *The Handbook of Road Safety Measures* (2nd ed.). Emerald Group Publishing.

##  Appendix

Run this code once after downloading all 3 csv files from:

* **Data source:** [City of Chicago — Traffic Crashes (Crashes)](https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if)

* **Data source:** [City of Chicago — Traffic Crashes (People)](https://data.cityofchicago.org/Transportation/Traffic-Crashes-People/u6pd-qa9d)

* **Data source:** [City of Chicago — Traffic Crashes (Vehicles)](https://data.cityofchicago.org/Transportation/Traffic-Crashes-Vehicles/68nd-jvt3)

* **Data source:** [City of Chicago — Street Center Lines](https://data.cityofchicago.org/Transportation/Street-Center-Lines/6imu-meau)

Then adjust the filepaths and run once, before running the notebook.

In [25]:
# import json
# import re
# import numpy as np
# import pandas as pd
# import plotly.colors as pc
# import plotly.graph_objects as go
# from shapely.geometry import LineString, MultiLineString, Point
# from shapely.strtree import STRtree

# ADJUST THE FILEPATHS HERE
# STREET_INFO_CSV = "street_geojson_info.csv"
# STREETS_GEOJSON = "transportation_20260416.geojson"

# streets_df = pd.read_csv(STREET_INFO_CSV)
# crashes_raw = pd.read_csv("Traffic_Crashes_-_Crashes_20260414.csv")
# vehicles_df = pd.read_csv("Traffic_Crashes_-_Vehicles_20260415.csv")
# people_df = pd.read_csv("Traffic_Crashes_-_People_20260415.csv")






# # CRASHES_CSV = "Traffic_Crashes_-_Crashes_20260414.csv"
# # COLOR_SCALE = "YlOrRd"
# MAX_MATCH_DIST_DEG = 0.00045
# MAX_MATCH_DIST_NAME_DEG = 0.0012

# def normalize_street_name(value):
#     if pd.isna(value):
#         return None
#     s = str(value).upper().strip()
#     s = re.sub(r"\s+", " ", s)
#     s = re.sub(r"[^A-Z0-9 ]", "", s)
#     return s or None


# def geometry_groups(geometry):
#     gtype = (geometry or {}).get("type")
#     coords = (geometry or {}).get("coordinates", [])
#     if gtype == "LineString":
#         return [coords]
#     if gtype == "MultiLineString":
#         return coords
#     return []


# def append_segment(target_lons, target_lats, segment):
#     if not segment:
#         return
#     for lon, lat in segment:
#         target_lons.append(lon)
#         target_lats.append(lat)
#     target_lons.append(None)
#     target_lats.append(None)


# def pick_street_name(row, name_cols):
#     for col in name_cols:
#         if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
#             return row[col]
#     return None


# def with_alpha(color_value, alpha):
#     m = re.match(r"rgb\((\d+),\s*(\d+),\s*(\d+)\)", str(color_value))
#     if m:
#         r, g, b = m.groups()
#         return f"rgba({r}, {g}, {b}, {alpha})"
#     return color_value

# print(streets_df.columns.to_list())

# crashes_df = people_df.merge(
#     crashes_raw,
#     on='CRASH_RECORD_ID',
#     how='left'
# ).merge(
#     vehicles_df[['CRASH_RECORD_ID', 'VEHICLE_ID', 'VEHICLE_TYPE', 'NUM_PASSENGERS', 
#                  'OCCUPANT_CNT', 'EXCEED_SPEED_LIMIT_I', 'MANEUVER', 'TOWED_I', 
#                  'FIRE_I', 'VEHICLE_DEFECT', 'VEHICLE_USE', 'TRAVEL_DIRECTION']],
#     on=['CRASH_RECORD_ID', 'VEHICLE_ID'],
#     how='left'  # Keep people even if they have NaN VEHICLE_ID (pedestrians/cyclists)
# )
# crashes_df['IS_PEDESTRIAN_CYCLIST'] = crashes_df['VEHICLE_ID'].isna()

# with open(STREETS_GEOJSON, "r", encoding="utf-8") as f:
#     geojson_streets = json.load(f)

# features = geojson_streets.get("features", [])
# if not features:
#     raise ValueError("GeoJSON has no features.")

# # Filter crashes to those with valid dates and in the target year (2025)
# crashes_df["CRASH_DATE_PARSED"] = pd.to_datetime(crashes_df["CRASH_DATE_x"], errors="coerce")
# crashes_df = crashes_df[crashes_df["CRASH_DATE_PARSED"].notna()].copy()
# crashes_df["CRASH_YEAR"] = crashes_df["CRASH_DATE_PARSED"].dt.year.astype(str)

# available_years = sorted(crashes_df["CRASH_YEAR"].dropna().unique().tolist())
# if not available_years:
#     raise ValueError("No valid crash years found in CRASH_DATE.")

# initial_year = max(available_years)

# street_candidates = ["STREET_NAME", "ON_STREET_NAME", "STREET_NAM", "STREET", "STREET_NO"]
# person_street_col = next((c for c in street_candidates if c in crashes_df.columns), None)
# if person_street_col is None:
#     raise KeyError(f"No street column found in crashes data. Tried: {street_candidates}")

# geojson_name_candidates = [
#     "STREET_NAME",
#     "STREET_NAM",
#     "STREETNAME",
#     "FULL_STREET",
#     "FULLNAME",
#     "ST_NAME",
#     "STREET",
#     "RD_NAME",
#     "NAME",
# ]
# csv_name_candidates = ["STREET_NAME", "STREET_NAM", "STREETNAME", "FULL_STREET", "STREET"]
# link_candidates = ["TRANS_ID", "OBJECTID", "FNODE_ID", "TNODE_ID"]

# # Build a feature-property frame and merge to street info via a shared key.
# feature_rows = []
# for idx, feature in enumerate(features):
#     props = feature.get("properties", {}) or {}
#     row = {"FEATURE_IDX": idx}
#     for k, v in props.items():
#         row[str(k).upper()] = v
#     feature_rows.append(row)

# feature_df = pd.DataFrame(feature_rows)
# streets_df_norm = streets_df.copy()
# streets_df_norm.columns = [str(c).upper() for c in streets_df_norm.columns]

# merge_key = next(
#     (c for c in link_candidates if c in feature_df.columns and c in streets_df_norm.columns),
#     None,
# )

# if merge_key is not None:
#     feature_df[merge_key] = feature_df[merge_key].astype(str)
#     streets_df_norm[merge_key] = streets_df_norm[merge_key].astype(str)
#     feature_street_df = feature_df.merge(
#         streets_df_norm,
#         on=merge_key,
#         how="left",
#         suffixes=("_GEO", "_CSV"),
#     )
# else:
#     feature_street_df = feature_df.copy()

# geo_name_cols = [f"{c}_GEO" for c in geojson_name_candidates if f"{c}_GEO" in feature_street_df.columns]
# csv_name_cols = [f"{c}_CSV" for c in csv_name_candidates if f"{c}_CSV" in feature_street_df.columns]

# if not geo_name_cols:
#     geo_name_cols = [c for c in geojson_name_candidates if c in feature_street_df.columns]
# if not csv_name_cols:
#     csv_name_cols = [c for c in csv_name_candidates if c in feature_street_df.columns]

# feature_street_df["STREET_RAW"] = feature_street_df.apply(
#     lambda r: pick_street_name(r, geo_name_cols + csv_name_cols),
#     axis=1,
# )
# feature_street_df["STREET_KEY"] = feature_street_df["STREET_RAW"].map(normalize_street_name)

# crash_type_candidates = ["CRASH_TYPE", "FIRST_CRASH_TYPE", "PRIM_CONTRIBUTORY_CAUSE"]
# crash_type_col = next((c for c in crash_type_candidates if c in crashes_df.columns), None)

# crash_points_cols = ["CRASH_YEAR", person_street_col, "LATITUDE", "LONGITUDE"]
# if crash_type_col is not None:
#     crash_points_cols.append(crash_type_col)

# crash_points_df = crashes_df[crash_points_cols].dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
# crash_points_df["LATITUDE"] = crash_points_df["LATITUDE"].astype(float)
# crash_points_df["LONGITUDE"] = crash_points_df["LONGITUDE"].astype(float)
# crash_points_df["CRASH_STREET_KEY"] = crash_points_df[person_street_col].map(normalize_street_name)
# if crash_type_col is not None:
#     crash_points_df["CRASH_TYPE_HOVER"] = crash_points_df[crash_type_col].fillna("Unknown")
# else:
#     crash_points_df["CRASH_TYPE_HOVER"] = "Unknown"

# feature_geometries = []
# valid_geom_indices = []
# valid_geoms = []

# for idx, feature in enumerate(features):
#     lines = []
#     for coords in geometry_groups(feature.get("geometry", {})):
#         if len(coords) >= 2:
#             lines.append(LineString(coords))

#     if not lines:
#         feature_geometries.append(None)
#         continue

#     geom = lines[0] if len(lines) == 1 else MultiLineString(lines)
#     feature_geometries.append(geom)
#     valid_geom_indices.append(idx)
#     valid_geoms.append(geom)

# if not valid_geoms:
#     raise ValueError("No valid street geometries available for matching.")

# street_tree = STRtree(valid_geoms)
# wkb_to_tree_pos = {geom.wkb: i for i, geom in enumerate(valid_geoms)}
# street_key_to_tree_positions = {}
# for tree_pos, feature_idx in enumerate(valid_geom_indices):
#     key = feature_street_df.iloc[feature_idx]["STREET_KEY"] if feature_idx < len(feature_street_df) else None
#     if key is None or pd.isna(key):
#         continue
#     street_key_to_tree_positions.setdefault(key, []).append(tree_pos)

# feature_counts = [0] * len(features)
# assigned_rows = []
# print(crash_points_df.columns.to_list())

# for source_idx, p_row in crash_points_df.iterrows():
#     year = int(p_row["CRASH_YEAR"])
#     lon = float(p_row["LONGITUDE"])
#     lat = float(p_row["LATITUDE"])
#     crash_type = str(p_row["CRASH_TYPE_HOVER"])
#     crash_street_key = p_row["CRASH_STREET_KEY"]
#     p = Point(lon, lat)

#     best_feature_idx = -1
#     match_method = "none"
#     best_dist = float("inf")

#     candidate_positions = street_key_to_tree_positions.get(crash_street_key, []) if crash_street_key else []
#     if candidate_positions:
#         for tree_pos in candidate_positions:
#             d = p.distance(valid_geoms[tree_pos])
#             if d < best_dist:
#                 best_dist = d
#                 best_feature_idx = valid_geom_indices[tree_pos]
#         if best_feature_idx >= 0 and best_dist <= MAX_MATCH_DIST_NAME_DEG:
#             match_method = "name+distance"
#             assigned_rows.append((source_idx, year, lon, lat, best_feature_idx, crash_type, match_method))
#             continue

#     nearest = street_tree.nearest(p)

#     if nearest is None:
#         assigned_rows.append((source_idx, year, lon, lat, -1, crash_type, match_method))
#         continue

#     if isinstance(nearest, (int, np.integer)):
#         tree_pos = int(nearest)
#         nearest_geom = valid_geoms[tree_pos]
#     else:
#         nearest_geom = nearest
#         tree_pos = wkb_to_tree_pos.get(nearest_geom.wkb)
#         if tree_pos is None:
#             assigned_rows.append((source_idx, year, lon, lat, -1, crash_type, match_method))
#             continue

#     if p.distance(nearest_geom) <= MAX_MATCH_DIST_DEG:
#         feature_idx = valid_geom_indices[tree_pos]
#         match_method = "global_nearest"
#         assigned_rows.append((source_idx, year, lon, lat, feature_idx, crash_type, match_method))
#     else:
#         assigned_rows.append((source_idx, year, lon, lat, -1, crash_type, match_method))

# assigned_df = pd.DataFrame(
#     assigned_rows,
#     columns=["SOURCE_IDX", "CRASH_YEAR", "LONGITUDE", "LATITUDE", "FEATURE_IDX", "CRASH_TYPE_HOVER", "MATCH_METHOD"],
# )

# print(feature_street_df.columns.tolist())

# matched_meta_df = assigned_df[assigned_df["FEATURE_IDX"] >= 0].copy()
# matched_meta_df["FEATURE_IDX"] = matched_meta_df["FEATURE_IDX"].astype(int)

# feature_length_lookup = feature_street_df.set_index("FEATURE_IDX")["LENGTH_CSV"]
# matched_meta_df["STREET_LENGTH"] = matched_meta_df["FEATURE_IDX"].map(feature_length_lookup)

# print(matched_meta_df.columns.tolist())

# # matched_meta_df = assigned_df[assigned_df["FEATURE_IDX"] >= 0].copy()
# # matched_meta_df["FEATURE_IDX"] = matched_meta_df["FEATURE_IDX"].astype(int)
# unmatched_df = assigned_df[assigned_df["FEATURE_IDX"] < 0].copy()

# matched_df = crashes_df.loc[matched_meta_df["SOURCE_IDX"]].copy()
# matched_df["FEATURE_IDX"] = matched_meta_df["FEATURE_IDX"].to_numpy()
# matched_df["MATCH_METHOD"] = matched_meta_df["MATCH_METHOD"].to_numpy()
# matched_df["STREET_LENGTH"] = matched_meta_df["STREET_LENGTH"].to_numpy()
# matched_df = matched_df.reindex(
#     columns=list(crashes_df.columns) + ["FEATURE_IDX", "MATCH_METHOD", "STREET_LENGTH"]
# )

# print(feature_street_df[["FEATURE_IDX", merge_key, "LENGTH_CSV"]].head(20))
# print("Non-null LENGTH_CSV:", feature_street_df["LENGTH_CSV"].notna().sum())

# feature_idx_to_street = {
#     idx: (str(feature_street_df.iloc[idx]["STREET_RAW"]) if pd.notna(feature_street_df.iloc[idx]["STREET_RAW"]) else "Unknown")
#     for idx in range(len(features))
# }
# assigned_df["STREET_HOVER"] = assigned_df["FEATURE_IDX"].map(feature_idx_to_street).fillna("Unmatched")

# grouped_counts = (
#     matched_meta_df.groupby(["CRASH_YEAR", "FEATURE_IDX"]).size().rename("COUNT").reset_index()
# )

# feature_idx_to_street_key = {
#     idx: (feature_street_df.iloc[idx]["STREET_KEY"] if idx < len(feature_street_df) else None)
#     for idx in range(len(features))
# }

# feature_count_lookup = {
#     (str(row["CRASH_YEAR"]), int(row["FEATURE_IDX"])): int(row["COUNT"])
#     for _, row in grouped_counts.iterrows()
# }

# matched_df.to_csv("final.csv")